In [ ]:
# =============================================================================
# SETUP: Import libraries and define ALL shared functions
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# TAX CALCULATION FUNCTIONS (2024 MFJ Brackets)
# =============================================================================

STANDARD_DEDUCTION_MFJ = 30000  # Approximate 2025 standard deduction

def calculate_tax_mfj(taxable_income):
    """Calculate federal tax for Married Filing Jointly (2024 brackets)"""
    if taxable_income <= 0:
        return 0
    brackets = [
        (23200, 0.10), (94300, 0.12), (201050, 0.22),
        (383900, 0.24), (487450, 0.32), (731200, 0.35), (float('inf'), 0.37)
    ]
    tax = 0
    prev = 0
    for limit, rate in brackets:
        if taxable_income <= prev:
            break
        taxable_in_bracket = min(taxable_income, limit) - prev
        tax += taxable_in_bracket * rate
        prev = limit
    return tax

def get_marginal_rate(taxable_income):
    """Get marginal tax rate for a given taxable income"""
    if taxable_income <= 0:
        return 0.10
    brackets = [
        (23200, 0.10), (94300, 0.12), (201050, 0.22),
        (383900, 0.24), (487450, 0.32), (731200, 0.35), (float('inf'), 0.37)
    ]
    for limit, rate in brackets:
        if taxable_income <= limit:
            return rate
    return 0.37

# =============================================================================
# RMD TABLE (IRS Uniform Lifetime Table)
# =============================================================================

RMD_DIVISORS = {
    72: 27.4, 73: 26.5, 74: 25.5, 75: 24.6, 76: 23.7, 77: 22.9, 78: 22.0,
    79: 21.1, 80: 20.2, 81: 19.4, 82: 18.5, 83: 17.7, 84: 16.8, 85: 16.0,
    86: 15.2, 87: 14.4, 88: 13.7, 89: 12.9, 90: 12.2, 91: 11.5, 92: 10.8,
    93: 10.1, 94: 9.5, 95: 8.9
}

def calculate_rmd(ira_balance, age):
    """Calculate Required Minimum Distribution"""
    if age < 73:
        return 0
    divisor = RMD_DIVISORS.get(age, 8.9)
    return ira_balance / divisor

# =============================================================================
# PROJECTION FUNCTION (used by all analysis cells)
# =============================================================================

def project_with_tax_tracking(
    annual_conversion,
    conversion_years,
    allow_32_bracket=False,
    # These will be set when called from cells that have the data
    total_pretax=None,
    total_roth=None,
    joint_taxable=None,
    spouse1_age=None,
    spouse2_age=None,
    spouse1_ss=None,
    spouse2_ss=None,
    years_to_s1_ss=None,
    years_to_s2_ss=None,
    annual_income=None,
    min_cash=None,
    ira_return=None,
    roth_return=None,
    taxable_return=None,
    inflation=None,
    # Monte Carlo (optional): per-year series. If provided, overrides scalar inputs above.
    horizon_years: int = 25,
    ira_returns=None,
    roth_returns=None,
    taxable_returns=None,
    inflation_rates=None,
    ): 
    """
    Project retirement path with detailed tax tracking.
    
    Parameters:
    -----------
    annual_conversion : float - Target annual conversion amount
    conversion_years : int - How many years to do conversions  
    allow_32_bracket : bool - If True, allow conversions into 32% bracket
    
    Monte Carlo extension (optional):
    - Provide `*_returns` arrays (length >= horizon_years) to simulate year-by-year returns.
    - Provide `inflation_rates` array (length >= horizon_years) to simulate year-by-year inflation.
    """
    
    if horizon_years <= 0:
        raise ValueError("horizon_years must be > 0")

    def _validate_series(name, series):
        if series is None:
            return
        if len(series) < horizon_years:
            raise ValueError(f"{name} must have length >= horizon_years ({horizon_years})")

    _validate_series("ira_returns", ira_returns)
    _validate_series("roth_returns", roth_returns)
    _validate_series("taxable_returns", taxable_returns)
    _validate_series("inflation_rates", inflation_rates)
    
    # Starting balances
    ira = total_pretax
    roth = total_roth
    taxable = joint_taxable
    
    # Tracking
    yearly_data = []
    cumulative_conv_tax = 0
    cumulative_rmd_tax = 0
    cumulative_total_tax = 0
    total_conversions = 0
    total_rmds = 0
    
    inflation_multiplier = 1.0
    
    for yr in range(horizon_years):
        rajesh_age = spouse1_age + yr
        terri_age = spouse2_age + yr
        
        # Use per-year simulated rates if provided
        ira_r = ira_returns[yr] if ira_returns is not None else ira_return
        roth_r = roth_returns[yr] if roth_returns is not None else roth_return
        taxable_r = taxable_returns[yr] if taxable_returns is not None else taxable_return
        infl_r = inflation_rates[yr] if inflation_rates is not None else inflation
        
        # === INCOME SOURCES ===
        ss1 = spouse1_ss if yr >= years_to_s1_ss else 0
        ss2 = spouse2_ss if yr >= years_to_s2_ss else 0
        total_ss = ss1 + ss2
        ss_taxable = total_ss * 0.85
        
        income_need = annual_income * inflation_multiplier
        from_savings_needed = max(0, income_need - total_ss)
        
        # === RMDs ===
        rajesh_rmd = calculate_rmd(ira * 0.33, rajesh_age)
        terri_rmd = calculate_rmd(ira * 0.67, terri_age)
        total_rmd = rajesh_rmd + terri_rmd
        total_rmds += total_rmd
        
        rmd_for_income = min(total_rmd, from_savings_needed)
        remaining_need = from_savings_needed - rmd_for_income
        
        # === WITHDRAWALS ===
        from_taxable = min(remaining_need * 0.5, max(0, taxable - min_cash))
        from_roth = min(remaining_need * 0.3, roth)
        from_ira_extra = max(0, remaining_need - from_taxable - from_roth)
        total_ira_withdrawal = total_rmd + from_ira_extra
        
        # === BASE TAXABLE INCOME (before conversions) ===
        base_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
        base_taxable_income = max(0, base_taxable_income)
        
        # === ROTH CONVERSIONS ===
        conversion = 0
        conversion_tax = 0
        conversion_bracket = 0
        
        if yr < conversion_years and annual_conversion > 0:
            available_for_conv_tax = max(0, taxable - min_cash - from_taxable)
            
            # Bracket limits (MFJ 2024)
            bracket_24_ceiling = 383900
            bracket_32_ceiling = 487450
            
            if allow_32_bracket:
                room_in_brackets = max(0, bracket_32_ceiling - base_taxable_income)
            else:
                room_in_brackets = max(0, bracket_24_ceiling - base_taxable_income)
            
            max_affordable = available_for_conv_tax / 0.28 if available_for_conv_tax > 0 else 0
            conversion = min(annual_conversion, room_in_brackets, max_affordable, ira - total_ira_withdrawal)
            conversion = max(0, conversion)
            
            if conversion > 0:
                income_after_conv = base_taxable_income + conversion
                conversion_tax = calculate_tax_mfj(income_after_conv) - calculate_tax_mfj(base_taxable_income)
                
                if base_taxable_income + conversion > bracket_24_ceiling:
                    conversion_bracket = 32
                elif base_taxable_income + conversion > 201050:
                    conversion_bracket = 24
                else:
                    conversion_bracket = 22
                
                total_conversions += conversion
                cumulative_conv_tax += conversion_tax
        
        # === RMD TAX ===
        total_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
        total_taxable_income = max(0, total_taxable_income)
        income_tax = calculate_tax_mfj(total_taxable_income)
        
        if total_ira_withdrawal > 0:
            rmd_share = total_rmd / total_ira_withdrawal
            rmd_tax = income_tax * rmd_share
        else:
            rmd_tax = 0
        
        cumulative_rmd_tax += rmd_tax
        cumulative_total_tax = cumulative_conv_tax + cumulative_rmd_tax
        
        # === UPDATE BALANCES ===
        ira -= total_ira_withdrawal
        ira -= conversion
        roth += conversion
        roth -= from_roth
        taxable -= from_taxable
        taxable -= conversion_tax
        taxable -= income_tax * 0.5
        
        ira *= (1 + ira_r)
        roth *= (1 + roth_r)
        taxable *= (1 + taxable_r)
        
        ira = max(0, ira)
        roth = max(0, roth)
        taxable = max(0, taxable)
        
        yearly_data.append({
            'year': yr + 1,
            'calendar_year': 2025 + yr,
            'rajesh_age': rajesh_age,
            'terri_age': terri_age,
            'conversion': conversion,
            'conversion_tax': conversion_tax,
            'conversion_bracket': conversion_bracket,
            'rmd': total_rmd,
            'rmd_tax': rmd_tax,
            'cumulative_conv_tax': cumulative_conv_tax,
            'cumulative_rmd_tax': cumulative_rmd_tax,
            'cumulative_total_tax': cumulative_total_tax,
            'ira': ira,
            'roth': roth,
            'taxable': taxable
        })
        
        # Update inflation multiplier at end of year so year 0 uses today's dollars
        inflation_multiplier *= (1 + infl_r)
    
    after_tax = ira * 0.75 + roth + taxable * 0.92
    legacy = ira * (1 - 0.28) + roth + taxable * 0.95
    
    return {
        'total_conversions': total_conversions,
        'total_conv_tax': cumulative_conv_tax,
        'total_rmds': total_rmds,
        'total_rmd_tax': cumulative_rmd_tax,
        'total_lifetime_tax': cumulative_total_tax,
        'after_tax': after_tax,
        'legacy': legacy,
        'yearly_data': yearly_data
    }

print("✅ Libraries and all shared functions loaded successfully")

---
# 📍 Chapter 1: Where You Stand Today
---

## ✅ Template: Enter inputs once
Edit the values in the next cell (the `INPUTS` dictionary). Everything else in the notebook reads from those values.

Tip: after changing inputs, use **Run All** to recompute outputs.

In [ ]:
# =============================================================================
# INPUTS (SAMPLE DATA): edit ONLY the INPUTS dictionary below
# =============================================================================

# Sample placeholder values:
# - Change names, ages, balances, and SS settings to match your household
# - Everything else in the notebook reads from these inputs

INPUTS = {
    # Household info
    "household": {
        "tax_filing_status": "MFJ",  # this notebook currently assumes Married Filing Jointly brackets
        "start_year": 2025,
    },
    # Spouse 1 (NOT yet at RMD age)
    "spouse1": {
        "name": "Spouse 1",
        "age": 70,
        "traditional_ira": 900_000,
        "sep_ira": 0,
        "roth_ira": 50_000,
        "ss_start_age": 70,
        "ss_annual": 48_000,
    },
    # Spouse 2 (already at/over RMD age)
    "spouse2": {
        "name": "Spouse 2",
        "age": 75,
        "traditional_ira": 1_150_000,
        "sep_ira": 0,
        "roth_ira": 100_000,
        "ss_start_age": 68,
        "ss_annual": 28_000,
    },
    # Joint accounts
    "joint": {
        "taxable_accounts": 450_000,
    },
    # Spending / cash
    "plan": {
        "monthly_income_need": 10_000,
        "minimum_cash_reserve": 75_000,
    },
    # Assumptions
    "assumptions": {
        "inflation_rate": 0.025,
        "taxable_return": 0.060,
        "ira_return": 0.060,
        "roth_return": 0.070,
    },
}

# =============================================================================
# VALIDATION (light sanity checks)
# =============================================================================

def _require_non_negative(value, label: str) -> None:
    if value is None:
        raise ValueError(f"{label} is missing")
    if value < 0:
        raise ValueError(f"{label} must be >= 0 (got {value})")

def _require_positive(value, label: str) -> None:
    if value is None:
        raise ValueError(f"{label} is missing")
    if value <= 0:
        raise ValueError(f"{label} must be > 0 (got {value})")

def _require_non_empty(value, label: str) -> None:
    if value is None or (isinstance(value, str) and value.strip() == ""):
        raise ValueError(f"{label} is missing")

_require_non_empty(INPUTS["spouse1"]["name"], "spouse1.name")
_require_positive(INPUTS["spouse1"]["age"], "spouse1.age")
_require_non_negative(INPUTS["spouse1"]["traditional_ira"], "spouse1.traditional_ira")
_require_non_negative(INPUTS["spouse1"]["sep_ira"], "spouse1.sep_ira")
_require_non_negative(INPUTS["spouse1"]["roth_ira"], "spouse1.roth_ira")
_require_positive(INPUTS["spouse1"]["ss_start_age"], "spouse1.ss_start_age")
_require_non_negative(INPUTS["spouse1"]["ss_annual"], "spouse1.ss_annual")

_require_non_empty(INPUTS["spouse2"]["name"], "spouse2.name")
_require_positive(INPUTS["spouse2"]["age"], "spouse2.age")
_require_non_negative(INPUTS["spouse2"]["traditional_ira"], "spouse2.traditional_ira")
_require_non_negative(INPUTS["spouse2"]["sep_ira"], "spouse2.sep_ira")
_require_non_negative(INPUTS["spouse2"]["roth_ira"], "spouse2.roth_ira")
_require_positive(INPUTS["spouse2"]["ss_start_age"], "spouse2.ss_start_age")
_require_non_negative(INPUTS["spouse2"]["ss_annual"], "spouse2.ss_annual")

_require_non_negative(INPUTS["joint"]["taxable_accounts"], "joint.taxable_accounts")
_require_positive(INPUTS["plan"]["monthly_income_need"], "plan.monthly_income_need")
_require_non_negative(INPUTS["plan"]["minimum_cash_reserve"], "plan.minimum_cash_reserve")

_require_positive(INPUTS["assumptions"]["inflation_rate"], "assumptions.inflation_rate")
_require_positive(INPUTS["assumptions"]["taxable_return"], "assumptions.taxable_return")
_require_positive(INPUTS["assumptions"]["ira_return"], "assumptions.ira_return")
_require_positive(INPUTS["assumptions"]["roth_return"], "assumptions.roth_return")

# =============================================================================
# BACKWARD-COMPAT VARIABLES (so the rest of the notebook runs unchanged)
# =============================================================================

# --- SPOUSE 1 ---
SPOUSE1_NAME = INPUTS["spouse1"]["name"]
SPOUSE1_AGE = INPUTS["spouse1"]["age"]
SPOUSE1_TRADITIONAL_IRA = INPUTS["spouse1"]["traditional_ira"]
SPOUSE1_SEP_IRA = INPUTS["spouse1"]["sep_ira"]
SPOUSE1_ROTH_IRA = INPUTS["spouse1"]["roth_ira"]
SPOUSE1_SS_START_AGE = INPUTS["spouse1"]["ss_start_age"]
SPOUSE1_SS_ANNUAL = INPUTS["spouse1"]["ss_annual"]  # Social Security at claiming age

# --- SPOUSE 2 ---
SPOUSE2_NAME = INPUTS["spouse2"]["name"]
SPOUSE2_AGE = INPUTS["spouse2"]["age"]
SPOUSE2_TRADITIONAL_IRA = INPUTS["spouse2"]["traditional_ira"]
SPOUSE2_SEP_IRA = INPUTS["spouse2"]["sep_ira"]
SPOUSE2_ROTH_IRA = INPUTS["spouse2"]["roth_ira"]
SPOUSE2_SS_START_AGE = INPUTS["spouse2"]["ss_start_age"]
SPOUSE2_SS_ANNUAL = INPUTS["spouse2"]["ss_annual"]

# --- JOINT ACCOUNTS ---
JOINT_TAXABLE_ACCOUNTS = INPUTS["joint"]["taxable_accounts"]

# --- YOUR NEEDS ---
MONTHLY_INCOME_NEED = INPUTS["plan"]["monthly_income_need"]
ANNUAL_INCOME_NEED = MONTHLY_INCOME_NEED * 12
MINIMUM_CASH_RESERVE = INPUTS["plan"]["minimum_cash_reserve"]

# --- ASSUMPTIONS ---
INFLATION_RATE = INPUTS["assumptions"]["inflation_rate"]
TAXABLE_RETURN = INPUTS["assumptions"]["taxable_return"]
IRA_RETURN = INPUTS["assumptions"]["ira_return"]
ROTH_RETURN = INPUTS["assumptions"]["roth_return"]

# Calculate totals
SPOUSE1_PRETAX = SPOUSE1_TRADITIONAL_IRA + SPOUSE1_SEP_IRA
SPOUSE2_PRETAX = SPOUSE2_TRADITIONAL_IRA + SPOUSE2_SEP_IRA
TOTAL_PRETAX = SPOUSE1_PRETAX + SPOUSE2_PRETAX
TOTAL_ROTH = SPOUSE1_ROTH_IRA + SPOUSE2_ROTH_IRA
TOTAL_WEALTH = TOTAL_PRETAX + TOTAL_ROTH + JOINT_TAXABLE_ACCOUNTS

print("═" * 80)
print("📍 CHAPTER 1: WHERE YOU STAND TODAY")
print("═" * 80)
print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         HOUSEHOLD BALANCE SHEET                             │
└─────────────────────────────────────────────────────────────────────────────┘

  👨 {SPOUSE1_NAME} (Age {SPOUSE1_AGE})
     ├── Traditional IRA:  ${SPOUSE1_TRADITIONAL_IRA:>12,.0f}
     ├── SEP IRA:          ${SPOUSE1_SEP_IRA:>12,.0f}
     ├── Roth IRA:         ${SPOUSE1_ROTH_IRA:>12,.0f}
     └── SUBTOTAL:         ${SPOUSE1_PRETAX:>12,.0f}

  👩 {SPOUSE2_NAME} (Age {SPOUSE2_AGE})
     ├── Traditional IRA:  ${SPOUSE2_TRADITIONAL_IRA:>12,.0f}
     ├── SEP IRA:          ${SPOUSE2_SEP_IRA:>12,.0f}
     ├── Roth IRA:         ${SPOUSE2_ROTH_IRA:>12,.0f}
     └── SUBTOTAL:         ${SPOUSE2_PRETAX:>12,.0f}

  🏦 Joint Taxable:        ${JOINT_TAXABLE_ACCOUNTS:>12,.0f}

  ════════════════════════════════════════════════════════════════════
  💰 TOTAL WEALTH:         ${TOTAL_WEALTH:>12,.0f}
  ════════════════════════════════════════════════════════════════════

  📊 ASSET ALLOCATION:
     • Pre-Tax (IRA/SEP):  ${TOTAL_PRETAX:>10,.0f}  ({TOTAL_PRETAX/TOTAL_WEALTH:.0%}) ← Uncle Sam's share here
     • Tax-Free (Roth):    ${TOTAL_ROTH:>10,.0f}  ({TOTAL_ROTH/TOTAL_WEALTH:.0%})
     • Taxable:            ${JOINT_TAXABLE_ACCOUNTS:>10,.0f}  ({JOINT_TAXABLE_ACCOUNTS/TOTAL_WEALTH:.0%})

  ⚠️  THE PROBLEM: {TOTAL_PRETAX/TOTAL_WEALTH:.0%} of your wealth is in PRE-TAX accounts.
     Every dollar withdrawn will be taxed as ordinary income!
""")

---
# 💵 Chapter 2: Your Income Needs & Timeline
---

In [ ]:
print("═" * 80)
print("💵 CHAPTER 2: YOUR INCOME NEEDS & TIMELINE")
print("═" * 80)

# Key milestones
years_to_terri_ss = SPOUSE2_SS_START_AGE - SPOUSE2_AGE
years_to_rajesh_ss = SPOUSE1_SS_START_AGE - SPOUSE1_AGE
years_to_terri_rmd = 73 - SPOUSE2_AGE
years_to_rajesh_rmd = 73 - SPOUSE1_AGE

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         YOUR INCOME REQUIREMENTS                            │
└─────────────────────────────────────────────────────────────────────────────┘

  📋 MONTHLY NEED:    ${MONTHLY_INCOME_NEED:>8,.0f}
  📋 ANNUAL NEED:     ${ANNUAL_INCOME_NEED:>8,.0f}
  📋 CASH RESERVE:    ${MINIMUM_CASH_RESERVE:>8,.0f} (emergency fund)

┌─────────────────────────────────────────────────────────────────────────────┐
│                         YOUR RETIREMENT TIMELINE                            │
└─────────────────────────────────────────────────────────────────────────────┘

  📅 TODAY (2025)
     {SPOUSE1_NAME} is {SPOUSE1_AGE}, {SPOUSE2_NAME} is {SPOUSE2_AGE}
     All income comes from savings (${ANNUAL_INCOME_NEED:,}/year)

  📅 YEAR {years_to_terri_ss} ({2025 + years_to_terri_ss}): {SPOUSE2_NAME}'s Social Security Starts
     └── +${SPOUSE2_SS_ANNUAL:,}/year

  📅 YEAR {years_to_rajesh_ss} ({2025 + years_to_rajesh_ss}): {SPOUSE1_NAME}'s Social Security Starts
     └── +${SPOUSE1_SS_ANNUAL:,}/year
     └── Combined SS: ${SPOUSE1_SS_ANNUAL + SPOUSE2_SS_ANNUAL:,}/year

  📅 YEAR {years_to_terri_rmd} ({2025 + years_to_terri_rmd}): {SPOUSE2_NAME} Turns 73 - RMDs BEGIN!
     └── IRS forces you to withdraw from IRAs
     └── All withdrawals taxed as ordinary income

  📅 YEAR {years_to_rajesh_rmd} ({2025 + years_to_rajesh_rmd}): {SPOUSE1_NAME} Turns 73 - More RMDs

  ⚠️  THE WINDOW: You have {years_to_terri_rmd} years before forced RMDs begin.
     These years are your OPPORTUNITY to convert to Roth strategically!
""")

---
# 📊 Chapter 3: THE KEY QUESTION - Is Paying 32% Worth It?
---

In [ ]:
# =============================================================================
# 📊 THE KEY QUESTION: Is Paying 32% Now Worth It?
# =============================================================================
# 
# If I convert aggressively and pay 32% tax now, will the savings from 
# lower RMDs later make up for it? And when does breakeven occur?
# =============================================================================

print("═" * 80)
print("📊 THE KEY QUESTION: IS PAYING 32% NOW WORTH IT?")
print("═" * 80)

# Helper function that uses the shared projection function with your data
def run_scenario(annual_conv, years, allow_32=False):
    return project_with_tax_tracking(
        annual_conversion=annual_conv,
        conversion_years=years,
        allow_32_bracket=allow_32,
        total_pretax=TOTAL_PRETAX,
        total_roth=TOTAL_ROTH,
        joint_taxable=JOINT_TAXABLE_ACCOUNTS,
        spouse1_age=SPOUSE1_AGE,
        spouse2_age=SPOUSE2_AGE,
        spouse1_ss=SPOUSE1_SS_ANNUAL,
        spouse2_ss=SPOUSE2_SS_ANNUAL,
        years_to_s1_ss=years_to_rajesh_ss,
        years_to_s2_ss=years_to_terri_ss,
        annual_income=ANNUAL_INCOME_NEED,
        min_cash=MINIMUM_CASH_RESERVE,
        ira_return=IRA_RETURN,
        roth_return=ROTH_RETURN,
        taxable_return=TAXABLE_RETURN,
        inflation=INFLATION_RATE
    )

# Run scenarios
conservative = run_scenario(100_000, 5, allow_32=False)   # Stay ≤24%
aggressive = run_scenario(175_000, 8, allow_32=True)       # Allow 32%
very_aggressive = run_scenario(200_000, 10, allow_32=True) # Max 32%
do_nothing = run_scenario(0, 0)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    THE QUESTION YOU'RE REALLY ASKING                        │
└─────────────────────────────────────────────────────────────────────────────┘

  "If I pay 32% tax on conversions NOW, will the reduced RMDs LATER
   save me enough to make it worthwhile? And when does breakeven occur?"

┌─────────────────────────────────────────────────────────────────────────────┐
│                    SCENARIO COMPARISON                                      │
└─────────────────────────────────────────────────────────────────────────────┘

  ┌──────────────────────┬─────────────────┬─────────────────┬─────────────────┐
  │                      │  CONSERVATIVE   │   AGGRESSIVE    │ VERY AGGRESSIVE │
  │                      │  (Stay ≤24%)    │  (Allow 32%)    │   (Max 32%)     │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ Conversion Strategy  │ $100K/yr × 5yr  │ $175K/yr × 8yr  │ $200K/yr × 10yr │
  │ Total Converted      │ ${conservative['total_conversions']:>13,.0f} │ ${aggressive['total_conversions']:>13,.0f} │ ${very_aggressive['total_conversions']:>13,.0f} │
  │ Conversion Tax Paid  │ ${conservative['total_conv_tax']:>13,.0f} │ ${aggressive['total_conv_tax']:>13,.0f} │ ${very_aggressive['total_conv_tax']:>13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ Total RMDs (25 yrs)  │ ${conservative['total_rmds']:>13,.0f} │ ${aggressive['total_rmds']:>13,.0f} │ ${very_aggressive['total_rmds']:>13,.0f} │
  │ Total RMD Tax        │ ${conservative['total_rmd_tax']:>13,.0f} │ ${aggressive['total_rmd_tax']:>13,.0f} │ ${very_aggressive['total_rmd_tax']:>13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ LIFETIME TAX         │ ${conservative['total_lifetime_tax']:>13,.0f} │ ${aggressive['total_lifetime_tax']:>13,.0f} │ ${very_aggressive['total_lifetime_tax']:>13,.0f} │
  │ vs Conservative      │ ${0:>13,.0f} │ ${aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:>+13,.0f} │ ${very_aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:>+13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ After-Tax Wealth     │ ${conservative['after_tax']:>13,.0f} │ ${aggressive['after_tax']:>13,.0f} │ ${very_aggressive['after_tax']:>13,.0f} │
  │ Legacy to Kids       │ ${conservative['legacy']:>13,.0f} │ ${aggressive['legacy']:>13,.0f} │ ${very_aggressive['legacy']:>13,.0f} │
  └──────────────────────┴─────────────────┴─────────────────┴─────────────────┘
""")

# Find breakeven point
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│              📈 BREAKEVEN ANALYSIS: WHEN DOES 32% PAY OFF?                  │
└─────────────────────────────────────────────────────────────────────────────┘
""")
print("  Year  Conservative         Aggressive           Difference")
print("        Cumul. Tax           Cumul. Tax           (Neg = Aggressive ahead)")
print("  ────  ───────────────────  ───────────────────  ─────────────────────")

crossover_year = None
for i in range(25):
    cons_data = conservative['yearly_data'][i]
    agg_data = aggressive['yearly_data'][i]
    diff = agg_data['cumulative_total_tax'] - cons_data['cumulative_total_tax']
    
    if diff < 0 and crossover_year is None:
        crossover_year = i + 1
        marker = " ← BREAKEVEN!"
    elif diff < 0:
        marker = " ✓"
    else:
        marker = ""
    
    if i < 12 or (crossover_year and abs(i + 1 - crossover_year) <= 2):
        print(f"  {i+1:>4}  ${cons_data['cumulative_total_tax']:>17,.0f}  ${agg_data['cumulative_total_tax']:>17,.0f}  ${diff:>+18,.0f}{marker}")

# The verdict
print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                       🎯 THE ANSWER                                         │
└─────────────────────────────────────────────────────────────────────────────┘
""")

if crossover_year:
    print(f"""
  ✅ YES, paying 32% CAN be worth it!
  
  ⏱️  BREAKEVEN: Year {crossover_year} ({2024 + crossover_year})
     That's when {SPOUSE1_NAME} is {SPOUSE1_AGE + crossover_year - 1} and {SPOUSE2_NAME} is {SPOUSE2_AGE + crossover_year - 1}
     
  After Year {crossover_year}, the aggressive strategy SAVES money every year
  because lower RMDs = less forced taxable income = less tax.
""")
else:
    print(f"""
  ❌ NO, paying 32% does NOT pay off in 25 years
  
  The extra tax paid upfront is NOT recovered through lower RMDs.
  Stick with the conservative (24% max) strategy.
""")

# Final comparison
print(f"""
  💰 25-YEAR OUTCOME COMPARISON:
  
     CONSERVATIVE (stay ≤24%):
     • Lifetime tax:      ${conservative['total_lifetime_tax']:>12,.0f}
     • After-tax wealth:  ${conservative['after_tax']:>12,.0f}
     • Legacy to kids:    ${conservative['legacy']:>12,.0f}
     
     AGGRESSIVE (allow 32%):
     • Lifetime tax:      ${aggressive['total_lifetime_tax']:>12,.0f}  ({aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:+,.0f})
     • After-tax wealth:  ${aggressive['after_tax']:>12,.0f}  ({aggressive['after_tax'] - conservative['after_tax']:+,.0f})
     • Legacy to kids:    ${aggressive['legacy']:>12,.0f}  ({aggressive['legacy'] - conservative['legacy']:+,.0f})

  🎯 RECOMMENDATION:
""")

if aggressive['after_tax'] > conservative['after_tax']:
    print(f"""     → AGGRESSIVE conversion is BETTER by ${aggressive['after_tax'] - conservative['after_tax']:,.0f}
     → The 32% tax is worth paying because RMD savings compound over time
     → Your kids also inherit ${aggressive['legacy'] - conservative['legacy']:,.0f} more (tax-free Roth)
""")
else:
    print(f"""     → CONSERVATIVE conversion is BETTER by ${conservative['after_tax'] - aggressive['after_tax']:,.0f}
     → The 32% tax is TOO expensive - stick with ≤24% conversions
     → Fill the 24% bracket fully, but don't spill into 32%
""")

---
# 🔀 Chapter 3: Your Three Paths
---

In [ ]:
print("═" * 80)
print("🔀 CHAPTER 3: YOUR THREE PATHS (WITH ACCURATE TAX CALCULATIONS)")
print("═" * 80)

# =============================================================================
# YEAR-BY-YEAR PROJECTION FUNCTION (local version for this analysis)
# Uses shared functions: calculate_tax_mfj, get_marginal_rate, calculate_rmd
# =============================================================================

def project_path(annual_conversion, conversion_years, path_name=""):
    """
    Project a path year-by-year with accurate tax calculations.
    
    This properly accounts for:
    - Income withdrawals from IRA (taxable)
    - Roth conversions (taxable, added on top of income)
    - Social Security (partially taxable - using 85%)
    - RMDs starting at age 73
    - Actual marginal tax rates based on combined income
    """
    
    # Starting balances
    ira = TOTAL_PRETAX
    roth = TOTAL_ROTH
    taxable = JOINT_TAXABLE_ACCOUNTS
    
    # Tracking
    total_conversions = 0
    total_conversion_tax = 0
    total_rmds = 0
    total_rmd_tax = 0
    yearly_details = []
    
    for yr in range(25):
        rajesh_age = SPOUSE1_AGE + yr
        terri_age = SPOUSE2_AGE + yr
        
        year_data = {
            'year': yr + 1,
            'calendar_year': 2025 + yr,
            'rajesh_age': rajesh_age,
            'terri_age': terri_age,
            'ira_start': ira,
            'roth_start': roth,
            'taxable_start': taxable
        }
        
        # === INCOME SOURCES ===
        
        # 1. Social Security (85% taxable for high earners)
        ss1 = SPOUSE1_SS_ANNUAL if yr >= years_to_rajesh_ss else 0
        ss2 = SPOUSE2_SS_ANNUAL if yr >= years_to_terri_ss else 0
        total_ss = ss1 + ss2
        ss_taxable = total_ss * 0.85  # 85% of SS is taxable at your income level
        
        # 2. Income need (inflation adjusted)
        income_need = ANNUAL_INCOME_NEED * (1 + INFLATION_RATE) ** yr
        
        # 3. How much do we need from savings after SS?
        from_savings_needed = max(0, income_need - total_ss)
        
        # 4. RMDs (calculated on prior year-end balance for simplicity)
        rajesh_rmd = calculate_rmd(ira * 0.33, rajesh_age)  # ~1/3 is Rajesh's
        terri_rmd = calculate_rmd(ira * 0.67, terri_age)    # ~2/3 is Terri's
        total_rmd = rajesh_rmd + terri_rmd
        total_rmds += total_rmd
        
        # RMD satisfies some of the income need
        rmd_for_income = min(total_rmd, from_savings_needed)
        remaining_need = from_savings_needed - rmd_for_income
        
        # === WHERE DOES INCOME COME FROM? ===
        
        # Priority: RMDs first, then taxable, then Roth, then additional IRA
        from_rmd = rmd_for_income
        from_taxable = min(remaining_need * 0.5, max(0, taxable - MINIMUM_CASH_RESERVE))
        from_roth = min(remaining_need * 0.3, roth)
        from_ira_extra = max(0, remaining_need - from_taxable - from_roth)
        
        # Total IRA withdrawal = RMD + any extra needed
        total_ira_withdrawal = total_rmd + from_ira_extra
        
        year_data['ss_income'] = total_ss
        year_data['rmd'] = total_rmd
        year_data['from_ira'] = total_ira_withdrawal
        year_data['from_taxable'] = from_taxable
        year_data['from_roth'] = from_roth
        
        # === ROTH CONVERSIONS ===
        
        conversion = 0
        conversion_tax = 0
        
        if yr < conversion_years and annual_conversion > 0:
            # How much can we afford to convert?
            available_for_conv_tax = max(0, taxable - MINIMUM_CASH_RESERVE - from_taxable)
            
            # What's our base taxable income BEFORE conversion?
            base_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
            base_taxable_income = max(0, base_taxable_income)
            
            # What marginal rate would we pay on conversions?
            marginal_rate = get_marginal_rate(base_taxable_income)
            
            # How much can we convert and stay in 24% bracket?
            room_in_24_bracket = max(0, 383900 - base_taxable_income)
            
            # Limit conversion to: available funds for tax, annual target, room in bracket, IRA balance
            max_affordable = available_for_conv_tax / marginal_rate if marginal_rate > 0 else 0
            conversion = min(annual_conversion, room_in_24_bracket, max_affordable, ira - total_ira_withdrawal)
            conversion = max(0, conversion)
            
            if conversion > 0:
                # Calculate actual tax on conversion (at marginal rate after other income)
                income_before_conv = base_taxable_income
                income_after_conv = base_taxable_income + conversion
                conversion_tax = calculate_tax_mfj(income_after_conv) - calculate_tax_mfj(income_before_conv)
                
                total_conversions += conversion
                total_conversion_tax += conversion_tax
        
        year_data['conversion'] = conversion
        year_data['conversion_tax'] = conversion_tax
        
        # === TAX ON IRA WITHDRAWALS (income + RMDs) ===
        
        # Tax on regular income (IRA withdrawals that aren't conversions)
        total_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
        total_taxable_income = max(0, total_taxable_income)
        income_tax = calculate_tax_mfj(total_taxable_income)
        
        # RMD portion of the tax
        if total_ira_withdrawal > 0:
            rmd_share = total_rmd / total_ira_withdrawal
            rmd_tax = income_tax * rmd_share
        else:
            rmd_tax = 0
        total_rmd_tax += rmd_tax
        
        year_data['income_tax'] = income_tax
        
        # === UPDATE BALANCES ===
        
        # Withdrawals
        ira -= total_ira_withdrawal
        ira -= conversion  # Move to Roth
        roth += conversion
        roth -= from_roth
        taxable -= from_taxable
        taxable -= conversion_tax  # Pay conversion tax from taxable
        taxable -= income_tax * 0.5  # Rough estimate: some tax paid from taxable
        
        # Growth
        ira *= (1 + IRA_RETURN)
        roth *= (1 + ROTH_RETURN)
        taxable *= (1 + TAXABLE_RETURN)
        
        # Ensure minimums
        ira = max(0, ira)
        roth = max(0, roth)
        taxable = max(0, taxable)
        
        year_data['ira_end'] = ira
        year_data['roth_end'] = roth
        year_data['taxable_end'] = taxable
        
        yearly_details.append(year_data)
    
    # Final calculations
    after_tax = ira * 0.75 + roth + taxable * 0.92
    legacy = ira * (1 - 0.28) + roth + taxable * 0.95
    first_rmd = yearly_details[years_to_terri_rmd - 1]['rmd'] if years_to_terri_rmd <= 25 else 0
    
    return {
        'path_name': path_name,
        'total_conversions': total_conversions,
        'total_conversion_tax': total_conversion_tax,
        'effective_conv_rate': (total_conversion_tax / total_conversions * 100) if total_conversions > 0 else 0,
        'total_rmds': total_rmds,
        'total_rmd_tax': total_rmd_tax,
        'ira': ira,
        'roth': roth,
        'taxable': taxable,
        'after_tax': after_tax,
        'legacy': legacy,
        'first_rmd': first_rmd,
        'yearly_details': yearly_details
    }

# =============================================================================
# RUN THE THREE PATHS
# =============================================================================

# PATH A: Do Nothing
path_a = project_path(annual_conversion=0, conversion_years=0, path_name="Do Nothing")

# PATH B: Smart Conversions ($100K/year for 5 years = $500K total)
path_b = project_path(annual_conversion=100_000, conversion_years=5, path_name="Smart Convert")

# PATH C: Aggressive Conversions ($150K/year for 10 years)
path_c = project_path(annual_conversion=150_000, conversion_years=10, path_name="Aggressive")

# =============================================================================
# DISPLAY RESULTS
# =============================================================================

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│          💡 HOW THIS ANALYSIS WORKS (NOW WITH ACCURATE TAXES!)              │
└─────────────────────────────────────────────────────────────────────────────┘

  This analysis now properly calculates taxes based on your ACTUAL situation:

  📊 YOUR ANNUAL INCOME STRUCTURE:
     ├── Living expenses needed:  ${ANNUAL_INCOME_NEED:>10,}/year
     ├── IRA withdrawals for living: TAXABLE as ordinary income
     ├── Roth conversions: ADDED to other income, taxed at marginal rate
     └── Standard deduction:      -${STANDARD_DEDUCTION_MFJ:>10,}

  ⚠️  THE KEY INSIGHT: 
     When you convert $100K to Roth, it's taxed ON TOP of your living expenses!
     
     Example Year 1 (before SS):
     • IRA withdrawal for living: ~$132,000 (your income need)
     • Roth conversion:           +$100,000
     • TOTAL taxable income:       $232,000 - $30K deduction = $202,000
     • Marginal rate on conversion: 24% (just into that bracket!)
""")

years_to_s1_rmd_local = max(0, 73 - SPOUSE1_AGE)
years_to_s2_rmd_local = max(0, 73 - SPOUSE2_AGE)
first_rmd_person = SPOUSE1_NAME if years_to_s1_rmd_local <= years_to_s2_rmd_local else SPOUSE2_NAME
first_rmd_label = f"First RMD ({first_rmd_person} age 73)"

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         THE THREE PATHS COMPARED                            │
└─────────────────────────────────────────────────────────────────────────────┘

╔═══════════════════════════════════════════════════════════════════════════════════════════╗
║                           ║   PATH A:          ║   PATH B:          ║   PATH C:           ║
║       METRIC              ║   Do Nothing       ║   Smart Convert    ║   Aggressive        ║
╠═══════════════════════════╬════════════════════╬════════════════════╬═════════════════════╣
║ Conversion Strategy       ║ $0/year × 0 yrs    ║ $100K/yr × 5 yrs   ║ $150K/yr × 10 yrs   ║
║ Total Conversions         ║ ${path_a['total_conversions']:>16,.0f}  ║ ${path_b['total_conversions']:>16,.0f}  ║ ${path_c['total_conversions']:>16,.0f}   ║
║ Total Conversion Tax      ║ ${path_a['total_conversion_tax']:>16,.0f}  ║ ${path_b['total_conversion_tax']:>16,.0f}  ║ ${path_c['total_conversion_tax']:>16,.0f}   ║
║ Effective Tax Rate        ║            {'N/A':>6}  ║ {path_b['effective_conv_rate']:>15.1f}%  ║ {path_c['effective_conv_rate']:>15.1f}%   ║
╠═══════════════════════════╬════════════════════╬════════════════════╬═════════════════════╣
║ Total RMDs (25 years)     ║ ${path_a['total_rmds']:>16,.0f}  ║ ${path_b['total_rmds']:>16,.0f}  ║ ${path_c['total_rmds']:>16,.0f}   ║
║ Total RMD Tax             ║ ${path_a['total_rmd_tax']:>16,.0f}  ║ ${path_b['total_rmd_tax']:>16,.0f}  ║ ${path_c['total_rmd_tax']:>16,.0f}   ║
╠═══════════════════════════╬════════════════════╬════════════════════╬═════════════════════╣
║ IRA at Year 25            ║ ${path_a['ira']:>16,.0f}  ║ ${path_b['ira']:>16,.0f}  ║ ${path_c['ira']:>16,.0f}   ║
║ Roth at Year 25           ║ ${path_a['roth']:>16,.0f}  ║ ${path_b['roth']:>16,.0f}  ║ ${path_c['roth']:>16,.0f}   ║
║ Taxable at Year 25        ║ ${path_a['taxable']:>16,.0f}  ║ ${path_b['taxable']:>16,.0f}  ║ ${path_c['taxable']:>16,.0f}   ║
╠═══════════════════════════╬════════════════════╬════════════════════╬═════════════════════╣
║ AFTER-TAX WEALTH          ║ ${path_a['after_tax']:>16,.0f}  ║ ${path_b['after_tax']:>16,.0f}  ║ ${path_c['after_tax']:>16,.0f}   ║
║ vs Do Nothing             ║ ${0:>16,.0f}  ║ ${path_b['after_tax'] - path_a['after_tax']:>+16,.0f}  ║ ${path_c['after_tax'] - path_a['after_tax']:>+16,.0f}   ║
╠═══════════════════════════╬════════════════════╬════════════════════╬═════════════════════╣
║ LEGACY TO KIDS            ║ ${path_a['legacy']:>16,.0f}  ║ ${path_b['legacy']:>16,.0f}  ║ ${path_c['legacy']:>16,.0f}   ║
║ {first_rmd_label:<23} ║ ${path_a['first_rmd']:>16,.0f}  ║ ${path_b['first_rmd']:>16,.0f}  ║ ${path_c['first_rmd']:>16,.0f}   ║
╚═══════════════════════════╩════════════════════╩════════════════════╩═════════════════════╝
""")

# Show year-by-year for Path B (the recommended path)
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│              📅 PATH B YEAR-BY-YEAR (First 10 Years)                        │
└─────────────────────────────────────────────────────────────────────────────┘
""")
print("  Year  Ages    SS Income    IRA Wdraw    Conversion   Conv Tax    Marginal")
print("  ────  ─────   ──────────   ──────────   ──────────   ─────────   ────────")

for yd in path_b['yearly_details'][:10]:
    ages = f"{yd['rajesh_age']}/{yd['terri_age']}"
    ss = f"${yd['ss_income']:>9,.0f}" if yd['ss_income'] > 0 else "        -"
    ira_w = f"${yd['from_ira']:>9,.0f}"
    conv = f"${yd['conversion']:>9,.0f}" if yd['conversion'] > 0 else "        -"
    conv_tax = f"${yd['conversion_tax']:>8,.0f}" if yd['conversion_tax'] > 0 else "       -"
    
    # Calculate marginal rate
    base_income = yd['ss_income'] * 0.85 + yd['from_ira'] - STANDARD_DEDUCTION_MFJ
    rate = get_marginal_rate(base_income + yd['conversion']) * 100
    rate_str = f"{rate:.0f}%"
    
    print(f"  {yd['year']:>4}  {ages:<6}  {ss}   {ira_w}   {conv}   {conv_tax}      {rate_str}")

print(f"""
  ═══════════════════════════════════════════════════════════════════════════
  PATH B TOTALS:
     • Total Converted:    ${path_b['total_conversions']:>12,.0f}
     • Total Conv Tax:     ${path_b['total_conversion_tax']:>12,.0f} (Effective rate: {path_b['effective_conv_rate']:.1f}%)
     • Total RMDs:         ${path_b['total_rmds']:>12,.0f}
  ═══════════════════════════════════════════════════════════════════════════
""")

# Key insights
print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         💡 KEY INSIGHTS                                     │
└─────────────────────────────────────────────────────────────────────────────┘

  📊 THE TAX BRACKETS THAT MATTER FOR YOU (2024 MFJ, after $30K deduction):
     ┌────────────────────────────────────────────────────────────────────────┐
     │  22% bracket:  $94,300 - $201,050   ← Your income lands here          │
     │  24% bracket:  $201,050 - $383,900  ← Conversions push you here       │
     │  32% bracket:  $383,900 - $487,450  ← DANGER ZONE - avoid this!       │
     └────────────────────────────────────────────────────────────────────────┘
     
     Your $132K income (after deduction: ~$102K) starts in the 22% bracket.
     Adding conversions pushes you into 24%. The goal is to FILL the 24% 
     bracket but NOT spill into 32%!
     
     💡 Room in 24% bracket = $383,900 - $102,000 = ~$282,000/year
        But limited by funds to pay tax and other constraints.

  🎯 BEST PATH: {'PATH B (Smart Convert)' if path_b['after_tax'] >= path_c['after_tax'] else 'PATH C (Aggressive)'}
     • After-tax wealth: ${max(path_b['after_tax'], path_c['after_tax']):,.0f}
     • Gain vs doing nothing: ${max(path_b['after_tax'], path_c['after_tax']) - path_a['after_tax']:+,.0f}

  ⚠️  WATCH OUT FOR:
     • Year {years_to_terri_ss + 1}+: {SPOUSE2_NAME}'s SS adds +${SPOUSE2_SS_ANNUAL:,}/yr taxable income → less room in 24%
     • Year {years_to_rajesh_ss + 1}+: {SPOUSE1_NAME}'s SS adds +${SPOUSE1_SS_ANNUAL:,}/yr more → bracket space shrinks further
     • Year {min(years_to_terri_rmd, years_to_rajesh_rmd) + 1}+: RMDs start → may push you toward 32% automatically!

  💰 THE STRATEGY:
     • Convert at 22-24% NOW to avoid paying 24-32% on RMDs LATER
     • Path B effective rate: {path_b['effective_conv_rate']:.1f}% 
     • Without conversions, RMDs will be taxed at 24%+ when SS is also flowing
""")

# Store results for later chapters
conversion_total_b = path_b['total_conversions']
conversion_tax_b = path_b['total_conversion_tax']
conversion_total_c = path_c['total_conversions']
conversion_tax_c = path_c['total_conversion_tax']
path_a_after_tax = path_a['after_tax']
path_b_after_tax = path_b['after_tax']
path_c_after_tax = path_c['after_tax']
path_a_first_rmd = path_a['first_rmd']
path_b_first_rmd = path_b['first_rmd']
path_c_first_rmd = path_c['first_rmd']
path_a_ira = path_a['ira']
path_b_ira = path_b['ira']
path_c_ira = path_c['ira']
path_a_roth = path_a['roth']
path_b_roth = path_b['roth']
path_c_roth = path_c['roth']
path_a_taxable = path_a['taxable']
path_b_taxable = path_b['taxable']
path_c_taxable = path_c['taxable']

---
# 💰 Chapter 4: Maximizing YOUR Wealth
---

In [ ]:
print("═" * 80)
print("💰 CHAPTER 4: MAXIMIZING YOUR WEALTH")
print("═" * 80)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    WHAT DOES "MAXIMIZE WEALTH" MEAN?                        │
└─────────────────────────────────────────────────────────────────────────────┘

There are TWO ways to think about maximizing wealth:

  1️⃣  MAXIMIZE SPENDABLE INCOME
      → Focus on after-tax dollars YOU can spend
      → Minimize lifetime taxes
      → Prioritize Roth (tax-free withdrawals)

  2️⃣  MAXIMIZE TOTAL ACCOUNT BALANCES  
      → Don't pay taxes until forced (RMDs)
      → Let pre-tax accounts grow tax-deferred
      → May result in higher total, but lower after-tax value

  For MOST retirees, Option 1 is better because:
  • You actually GET to use the money
  • Roth withdrawals don't trigger SS taxation
  • Roth isn't subject to RMDs
  • More flexibility in retirement

┌─────────────────────────────────────────────────────────────────────────────┐
│                    YOUR OPTIMAL STRATEGY FOR WEALTH                         │
└─────────────────────────────────────────────────────────────────────────────┘

  Based on your situation:
  • ${JOINT_TAXABLE_ACCOUNTS:,} in taxable (to pay conversion taxes)
  • ${ANNUAL_INCOME_NEED:,}/year income need
  • 12 years before RMDs

  ✅ RECOMMENDED: PATH B - SMART CONVERSIONS

     Convert ~$500,000 over 4-5 years
     ├── Pay ~$120,000 in conversion taxes (from taxable)
     ├── Move $500,000 to Roth (grows 100% tax-free)
     ├── Reduce first RMD by ~${path_a_first_rmd - path_b_first_rmd:,.0f}
     └── Gain ~${path_b_after_tax - path_a_after_tax:,.0f} in after-tax wealth

  📊 25-YEAR OUTCOME:
     • After-Tax Wealth: ${path_b_after_tax:,.0f}
     • vs Doing Nothing: +${path_b_after_tax - path_a_after_tax:,.0f}
""")

---
# 👨‍👩‍👧‍👦 Chapter 5: Maximizing Legacy for Your Kids
---

In [ ]:
print("═" * 80)
print("👨‍👩‍👧‍👦 CHAPTER 5: MAXIMIZING LEGACY FOR YOUR KIDS")
print("═" * 80)

# Calculate inheritance scenarios
# Assume you live to age 85 (25 years), kids inherit the rest

# What kids receive (after-tax value to THEM)
# IRA: Kids must withdraw within 10 years, taxed at THEIR rate (~25-32%)
# Roth: Kids withdraw tax-free within 10 years
# Taxable: Gets stepped-up basis, minimal tax

kids_tax_rate = 0.28  # Assume kids are in 28% marginal bracket

# PATH A inheritance
path_a_kids_ira = path_a_ira * (1 - kids_tax_rate)  # Kids pay tax
path_a_kids_roth = path_a_roth  # Tax-free to kids
path_a_kids_taxable = path_a_taxable * 0.95  # Stepped-up basis, minimal tax
path_a_kids_total = path_a_kids_ira + path_a_kids_roth + path_a_kids_taxable

# PATH B inheritance
path_b_kids_ira = path_b_ira * (1 - kids_tax_rate)
path_b_kids_roth = path_b_roth  # Tax-free!
path_b_kids_taxable = path_b_taxable * 0.95
path_b_kids_total = path_b_kids_ira + path_b_kids_roth + path_b_kids_taxable

# PATH C inheritance
path_c_kids_ira = path_c_ira * (1 - kids_tax_rate)
path_c_kids_roth = path_c_roth  # Tax-free!
path_c_kids_taxable = path_c_taxable * 0.95
path_c_kids_total = path_c_kids_ira + path_c_kids_roth + path_c_kids_taxable

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    THE INHERITANCE TAX PROBLEM                              │
└─────────────────────────────────────────────────────────────────────────────┘

  When your kids inherit your accounts, here's what happens:

  📜 INHERITED IRA (Traditional/SEP):
     • Kids MUST withdraw everything within 10 years (SECURE Act)
     • Every dollar is taxed as THEIR ordinary income
     • If they're working, this could push them into 32-37% brackets!
     • Result: Uncle Sam takes 25-35% of their inheritance

  📜 INHERITED ROTH IRA:
     • Kids MUST withdraw within 10 years
     • BUT... every dollar is 100% TAX-FREE!
     • No impact on their tax bracket
     • Result: Kids keep 100% of the inheritance

  📜 INHERITED TAXABLE ACCOUNT:
     • Gets "stepped-up" cost basis to current value
     • Kids only pay tax on gains AFTER inheritance
     • Result: Very tax-efficient transfer

  💡 KEY INSIGHT: Converting IRA → Roth means YOU pay the tax at 22-24%
     instead of your KIDS paying at 28-35%!

┌─────────────────────────────────────────────────────────────────────────────┐
│                    WHAT YOUR KIDS ACTUALLY RECEIVE                          │
└─────────────────────────────────────────────────────────────────────────────┘

  Assuming you both live to ~85, and kids are in 28% tax bracket:

╔═══════════════════════╦═══════════════════╦═══════════════════╦═══════════════════╗
║                       ║   PATH A:         ║   PATH B:         ║   PATH C:         ║
║   INHERITANCE         ║   Do Nothing      ║   Smart Convert   ║   Aggressive      ║
╠═══════════════════════╬═══════════════════╬═══════════════════╬═══════════════════╣
║ IRA (after kids' tax) ║ ${path_a_kids_ira:>14,.0f}  ║ ${path_b_kids_ira:>14,.0f}  ║ ${path_c_kids_ira:>14,.0f}  ║
║ Roth (100% tax-free)  ║ ${path_a_kids_roth:>14,.0f}  ║ ${path_b_kids_roth:>14,.0f}  ║ ${path_c_kids_roth:>14,.0f}  ║
║ Taxable (stepped-up)  ║ ${path_a_kids_taxable:>14,.0f}  ║ ${path_b_kids_taxable:>14,.0f}  ║ ${path_c_kids_taxable:>14,.0f}  ║
╠═══════════════════════╬═══════════════════╬═══════════════════╬═══════════════════╣
║ KIDS' AFTER-TAX TOTAL ║ ${path_a_kids_total:>14,.0f}  ║ ${path_b_kids_total:>14,.0f}  ║ ${path_c_kids_total:>14,.0f}  ║
║ vs Do Nothing         ║ ${0:>14,.0f}  ║ ${path_b_kids_total - path_a_kids_total:>+14,.0f}  ║ ${path_c_kids_total - path_a_kids_total:>+14,.0f}  ║
╚═══════════════════════╩═══════════════════╩═══════════════════╩═══════════════════╝

  🎯 TO MAXIMIZE LEGACY:
""")

best_for_kids = max(path_a_kids_total, path_b_kids_total, path_c_kids_total)
if best_for_kids == path_c_kids_total:
    best_path = "PATH C (Aggressive)"
    best_amount = path_c_kids_total
    extra = path_c_kids_total - path_a_kids_total
elif best_for_kids == path_b_kids_total:
    best_path = "PATH B (Smart)"
    best_amount = path_b_kids_total
    extra = path_b_kids_total - path_a_kids_total
else:
    best_path = "PATH A (Do Nothing)"
    best_amount = path_a_kids_total
    extra = 0

print(f"""     
     ✅ {best_path} leaves your kids ${best_amount:,.0f}
     ✅ That's ${extra:,.0f} MORE than doing nothing!
     
     The more you convert to Roth NOW:
     • The less tax burden your KIDS face
     • The more flexibility they have
     • The more they actually KEEP
""")

---
# 📅 Chapter 6: Your Year-by-Year Action Plan
---

In [ ]:
print("═" * 80)
print("📅 CHAPTER 6: YOUR YEAR-BY-YEAR ACTION PLAN")
print("═" * 80)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         THE CONVERSION CALENDAR                             │
└─────────────────────────────────────────────────────────────────────────────┘

  ══════════════════════════════════════════════════════════════════════════
  📅 YEARS 1-4 (2025-2028): THE GOLDEN WINDOW
  ══════════════════════════════════════════════════════════════════════════
  
  Ages: {SPOUSE1_AGE}-{SPOUSE1_AGE+3} / {SPOUSE2_AGE}-{SPOUSE2_AGE+3}
  Social Security: None yet
  Tax Situation: Lowest brackets available
  
  ✅ ACTION: Convert $100,000 - $150,000 per year
  
  WHY: 
  • No SS income filling up your brackets
  • Room to fill 22% and 24% brackets cheaply
  • Taxable account can cover taxes
  • Each dollar converted = 25+ years of tax-free growth
  
  YEAR-BY-YEAR:
  ┌────────┬────────────┬─────────────┬─────────────┬──────────────┐
  │ Year   │ Your Ages  │ Convert     │ Est. Tax    │ Taxable After│
  ├────────┼────────────┼─────────────┼─────────────┼──────────────┤
  │ 2025   │ 60/61      │ $125,000    │ $27,500     │ ~$412,000    │
  │ 2026   │ 61/62      │ $125,000    │ $27,500     │ ~$385,000    │
  │ 2027   │ 62/63      │ $125,000    │ $27,500     │ ~$358,000    │
  │ 2028   │ 63/64      │ $125,000    │ $27,500     │ ~$331,000    │
  └────────┴────────────┴─────────────┴─────────────┴──────────────┘
  SUBTOTAL: $500,000 converted, ~$110,000 in taxes

  ══════════════════════════════════════════════════════════════════════════
  📅 YEAR {years_to_terri_ss + 1} ({2025 + years_to_terri_ss}): {SPOUSE2_NAME.upper()}'S SS STARTS
  ══════════════════════════════════════════════════════════════════════════
  
  Ages: {SPOUSE1_AGE + years_to_terri_ss}/{SPOUSE2_AGE + years_to_terri_ss}
  Social Security: +${SPOUSE2_SS_ANNUAL:,}/year ({SPOUSE2_NAME})
  Tax Situation: Less room in lower brackets
  
  ⚠️ ACTION: Reduce conversions to $50,000 - $75,000
  
  WHY:
  • SS income now fills some of your bracket space
  • Converting too much pushes into higher brackets
  • Still valuable, but be more selective

  ══════════════════════════════════════════════════════════════════════════
  📅 YEARS {years_to_terri_ss + 2}-{years_to_terri_ss + 3} ({2025 + years_to_terri_ss + 1}-{2025 + years_to_terri_ss + 2}): TRANSITION PERIOD
  ══════════════════════════════════════════════════════════════════════════
  
  Ages: {SPOUSE1_AGE + years_to_terri_ss + 1}-{SPOUSE1_AGE + years_to_terri_ss + 2}/{SPOUSE2_AGE + years_to_terri_ss + 1}-{SPOUSE2_AGE + years_to_terri_ss + 2}
  Social Security: ${SPOUSE2_SS_ANNUAL:,}/year ({SPOUSE2_NAME})
  
  ⚠️ ACTION: Convert $25,000 - $50,000 if room in 24% bracket
  
  CHECKPOINT: Is your taxable account above $200,000?
  • YES → Continue modest conversions
  • NO → Pause and preserve flexibility

  ══════════════════════════════════════════════════════════════════════════
  📅 YEAR {max(years_to_rajesh_ss, years_to_terri_ss) + 1}+ ({2025 + max(years_to_rajesh_ss, years_to_terri_ss)}+): BOTH SS ACTIVE
  ══════════════════════════════════════════════════════════════════════════
  
  Ages: {SPOUSE1_AGE + max(years_to_rajesh_ss, years_to_terri_ss)}+/{SPOUSE2_AGE + max(years_to_rajesh_ss, years_to_terri_ss)}+
  Social Security: ${SPOUSE1_SS_ANNUAL + SPOUSE2_SS_ANNUAL:,}/year (both)
  
  🔍 ACTION: Evaluate annually - conversion may be minimal or stop
  
  ANNUAL CHECKLIST:
  □ Is taxable > $125,000?              Yes→ Consider | No→ Stop
  □ Will conversion stay in 24% bracket? Yes→ Convert  | No→ Skip
  □ Are RMDs at acceptable level?        No→ Convert   | Yes→ Optional

  ══════════════════════════════════════════════════════════════════════════
  📅 YEAR {min(years_to_rajesh_rmd, years_to_terri_rmd) + 1}+ ({2025 + min(years_to_rajesh_rmd, years_to_terri_rmd)}+): RMDs BEGIN
  ══════════════════════════════════════════════════════════════════════════
  
  Ages: {SPOUSE1_AGE + min(years_to_rajesh_rmd, years_to_terri_rmd)}/{SPOUSE2_AGE + min(years_to_rajesh_rmd, years_to_terri_rmd)}+
  Status: Required Minimum Distributions begin
  
  🛑 CONVERSION WINDOW CLOSES
  
  • RMDs add to taxable income automatically
  • Less room for conversions without hitting high brackets
  • Focus shifts to managing RMDs efficiently
""")

---
# 🎯 Chapter 7: The Bottom Line
---

In [ ]:
print("═" * 80)
print("🎯 CHAPTER 7: THE BOTTOM LINE")
print("═" * 80)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         SUMMARY: WHAT YOU SHOULD DO                         │
└─────────────────────────────────────────────────────────────────────────────┘

  ╔═══════════════════════════════════════════════════════════════════════════╗
  ║                     YOUR OPTIMAL STRATEGY                                 ║
  ╠═══════════════════════════════════════════════════════════════════════════╣
  ║                                                                           ║
  ║   Convert ~$500,000 to Roth over 4-5 years (2025-2029)                   ║
  ║                                                                           ║
  ║   • Annual conversion: ~$100,000 - $125,000                              ║
  ║   • Conversion taxes: ~$110,000 - $130,000 total                         ║
  ║   • Pay taxes from: Taxable account                                       ║
  ║                                                                           ║
  ╚═══════════════════════════════════════════════════════════════════════════╝

  📊 WHAT THIS ACHIEVES:

     ┌───────────────────────────────────┬──────────────────┐
     │ Metric                            │ Your Outcome     │
     ├───────────────────────────────────┼──────────────────┤
     │ After-tax wealth gain (25 yrs)    │ +${path_b_after_tax - path_a_after_tax:>13,.0f} │
     │ First RMD reduction               │ -${path_a_first_rmd - path_b_first_rmd:>13,.0f} │
     │ Kids' inheritance increase        │ +${path_b_kids_total - path_a_kids_total:>13,.0f} │
     └───────────────────────────────────┴──────────────────┘

  ✅ FOR MAXIMIZING YOUR WEALTH:
     → Convert steadily in Years 1-4 while in low brackets
     → Reduces lifetime taxes by paying 22-24% now vs 28-32% later
     → More tax-free income in retirement
     → More control (Roth has no RMDs for you)

  ✅ FOR MAXIMIZING LEGACY TO KIDS:
     → Every dollar in Roth = 100% to kids (tax-free)
     → Every dollar in IRA = 72% to kids (after their taxes)
     → You pay 24% tax now so kids avoid 28-35% tax later
     → Path B adds ${path_b_kids_total - path_a_kids_total:,.0f} to their inheritance

  🚫 WHAT NOT TO DO:
     ✗ Don't do nothing (costs ${path_a_after_tax - path_b_after_tax:,.0f} in wealth)
     ✗ Don't use IRA money to pay conversion taxes
     ✗ Don't convert so much you deplete taxable below $125K
     ✗ Don't convert into 32% bracket (rarely worth it)

  ════════════════════════════════════════════════════════════════════════════

  🎬 NEXT STEPS:

     1. Review this plan with your financial advisor / CPA
     2. Decide on Year 1 conversion amount ($100K-$150K)
     3. Execute conversion before December 31, 2025
     4. Set aside cash for April 2026 tax payment
     5. Repeat annually, adjusting as SS begins

  ════════════════════════════════════════════════════════════════════════════

  💝 Remember: The goal isn't just numbers—it's FREEDOM and SECURITY
     for {SPOUSE1_NAME} and {SPOUSE2_NAME} now, and a meaningful legacy for your kids.

═══════════════════════════════════════════════════════════════════════════════
""")

---
# 📋 Quick Reference Card
---

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                 📋 HOUSEHOLD RETIREMENT QUICK REFERENCE                       ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  YOUR NUMBERS:                                                                ║
║  ─────────────────────────────────────────────────────────────────────────    ║
""")
print(f"""║  Total Wealth:         ${TOTAL_WEALTH:>12,.0f}                                    ║""")
print(f"""║  Pre-Tax IRAs:         ${TOTAL_PRETAX:>12,.0f}                                    ║""")
print(f"""║  Taxable Account:      ${JOINT_TAXABLE_ACCOUNTS:>12,.0f}                                    ║""")
print(f"""║  Monthly Income Need:  ${MONTHLY_INCOME_NEED:>12,.0f}                                    ║""")
print(f"""║  Cash Reserve:         ${MINIMUM_CASH_RESERVE:>12,.0f}                                    ║""")
print("""
║                                                                               ║
║  CONVERSION TARGETS:                                                          ║
║  ─────────────────────────────────────────────────────────────────────────    ║
║  Years 1-4:  $100,000 - $150,000/year                                         ║
║  Year 5:     $50,000 - $75,000 (Spouse 2 SS starts)                           ║
║  Years 6-7:  $25,000 - $50,000 (if room)                                      ║
║  Year 8+:    Evaluate annually                                                ║
║                                                                               ║
║  STOP SIGNALS:                                                                ║
║  ─────────────────────────────────────────────────────────────────────────    ║
║  🛑 Taxable account < $125,000                                                ║
║  🛑 Conversion would push into 32% bracket                                    ║
║  🛑 RMDs already at acceptable level (<$100K/year)                            ║
║                                                                               ║
║  KEY DATES:                                                                   ║
║  ─────────────────────────────────────────────────────────────────────────    ║""")
print(f"""║  {2025 + years_to_terri_ss}: {SPOUSE2_NAME} SS starts (+${SPOUSE2_SS_ANNUAL:,}/year)                              ║""")
print(f"""║  {2025 + years_to_rajesh_ss}: {SPOUSE1_NAME} SS starts (+${SPOUSE1_SS_ANNUAL:,}/year)                             ║""")
print(f"""║  {2025 + years_to_terri_rmd}: {SPOUSE2_NAME} RMDs begin (age 73)                                      ║""")
print(f"""║  {2025 + years_to_rajesh_rmd}: {SPOUSE1_NAME} RMDs begin (age 73)                                     ║""")
print("""
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

---
# 🏠 Chapter 8: What If - Home Purchase ($200K Down Payment)
---

In [ ]:
# =============================================================================
# 🏠 CHAPTER 8: HOME PURCHASE SCENARIO (WITH RMD CALCULATIONS)
# =============================================================================
# This analysis now includes Required Minimum Distributions (RMDs) which
# start at age 73 for both spouses. RMDs are forced withdrawals from IRAs
# that add to your taxable income whether you need the money or not.
# =============================================================================

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │                    📝 CONFIGURABLE INPUT VARIABLES                         │
# └─────────────────────────────────────────────────────────────────────────────┘

# HOME_DOWN_PAYMENT: The amount needed for down payment
# - Typical range: $100K - $400K depending on home price
# - Higher = fewer funds for Roth conversion taxes
HOME_DOWN_PAYMENT = 200_000

# HOME_PURCHASE_YEAR: The calendar year you plan to buy (2025, 2026, 2027, etc.)
# - 2025: Buy immediately, fewest conversions before purchase
# - 2026: One year of conversions first, then buy
# - 2027+: More conversions before purchase, but longer wait
# 
# KEY INSIGHT: Each year you delay allows ~$100K-150K MORE in conversions
# because you still have full taxable account to pay conversion taxes.
HOME_PURCHASE_YEAR = 2027  # Enter the actual calendar year

# Calculate offset from base year (2025 = Year 1)
BASE_YEAR = 2025
PURCHASE_YEAR_OFFSET = HOME_PURCHASE_YEAR - BASE_YEAR  # 0 for 2025, 1 for 2026, etc.

# =============================================================================
# RMD TABLE (IRS Uniform Lifetime Table - ages 72-95)
# =============================================================================
RMD_DIVISORS = {
    72: 27.4, 73: 26.5, 74: 25.5, 75: 24.6, 76: 23.7, 77: 22.9, 78: 22.0,
    79: 21.1, 80: 20.2, 81: 19.4, 82: 18.5, 83: 17.7, 84: 16.8, 85: 16.0,
    86: 15.2, 87: 14.4, 88: 13.7, 89: 12.9, 90: 12.2, 91: 11.5, 92: 10.8,
    93: 10.1, 94: 9.5, 95: 8.9
}

def get_rmd_divisor(age):
    """Get RMD divisor for a given age"""
    if age < 73:
        return None  # No RMD required
    return RMD_DIVISORS.get(age, 8.9)  # Use 8.9 for ages 95+

def calculate_rmd(ira_balance, age):
    """Calculate Required Minimum Distribution"""
    divisor = get_rmd_divisor(age)
    if divisor is None:
        return 0
    return ira_balance / divisor

# =============================================================================
# ENHANCED PROJECTION FUNCTION WITH RMDs
# =============================================================================

def project_with_rmds(purchase_year_offset, down_payment, initial_ira, initial_roth, 
                      initial_taxable, conversion_years=5, max_annual_conv=150_000):
    """
    Enhanced projection that includes RMD calculations.
    
    Parameters:
    -----------
    purchase_year_offset : int
        Years from 2025 to buy home (0 = 2025, 1 = 2026, etc.)
    down_payment : float
        Down payment amount
    initial_ira, initial_roth, initial_taxable : float
        Starting balances
    conversion_years : int
        How many years to do conversions
    max_annual_conv : float
        Maximum conversion per year
    
    Returns detailed year-by-year breakdown including RMDs.
    """
    
    # Track balances
    ira = initial_ira
    roth = initial_roth
    taxable = initial_taxable
    
    # Results tracking
    total_conversions = 0
    total_rmds = 0
    total_rmd_tax = 0
    conversion_schedule = []
    rmd_schedule = []
    yearly_data = []
    
    # Ages at start (2025)
    rajesh_age_start = SPOUSE1_AGE  # 60
    terri_age_start = SPOUSE2_AGE   # 61
    
    # Split calculation for home purchase
    split_taxable = min(down_payment // 2, taxable - MINIMUM_CASH_RESERVE)
    split_ira_net = down_payment - split_taxable
    split_ira_gross = split_ira_net * 1.24  # Add 24% for tax
    
    for yr in range(25):
        rajesh_age = rajesh_age_start + yr
        terri_age = terri_age_start + yr
        year_data = {'year': yr + 1, 'calendar_year': 2025 + yr}
        
        # === BEGINNING OF YEAR ===
        year_data['ira_start'] = ira
        year_data['roth_start'] = roth
        year_data['taxable_start'] = taxable
        
        # === RMD CALCULATIONS ===
        # RMDs are based on prior year-end balance, but we'll use current for simplicity
        rajesh_rmd = calculate_rmd(ira * 0.33, rajesh_age)  # ~1/3 is Rajesh's
        terri_rmd = calculate_rmd(ira * 0.67, terri_age)    # ~2/3 is Terri's
        total_rmd_this_year = rajesh_rmd + terri_rmd
        
        # RMDs come out of IRA
        ira -= total_rmd_this_year
        total_rmds += total_rmd_this_year
        
        # RMD tax (at ~24% marginal rate)
        rmd_tax = total_rmd_this_year * 0.24
        total_rmd_tax += rmd_tax
        
        year_data['rajesh_age'] = rajesh_age
        year_data['terri_age'] = terri_age
        year_data['rmd'] = total_rmd_this_year
        year_data['rmd_tax'] = rmd_tax
        rmd_schedule.append(total_rmd_this_year)
        
        # === HOME PURCHASE ===
        home_this_year = (yr == purchase_year_offset)
        if home_this_year:
            taxable -= split_taxable
            ira -= split_ira_gross
            year_data['home_purchase'] = True
            year_data['home_from_taxable'] = split_taxable
            year_data['home_from_ira'] = split_ira_gross
        else:
            year_data['home_purchase'] = False
        
        # === ROTH CONVERSIONS ===
        conversion = 0
        if yr < conversion_years:
            # How much can we convert?
            available_for_tax = taxable - MINIMUM_CASH_RESERVE
            
            # Before home purchase, also reserve the down payment
            if yr < purchase_year_offset:
                available_for_tax -= down_payment
            
            if available_for_tax > 0:
                # Max conversion based on available tax funds
                max_from_funds = available_for_tax / 0.24
                
                # Reduced conversion in home purchase year (IRA withdrawal uses bracket space)
                if home_this_year:
                    max_this_year = 75_000
                else:
                    max_this_year = max_annual_conv
                
                conversion = min(max_from_funds, max_this_year, ira)
                conversion = max(0, conversion)
        
        # Execute conversion
        if conversion > 0:
            conv_tax = conversion * 0.24
            ira -= conversion
            roth += conversion
            taxable -= conv_tax
            total_conversions += conversion
        
        conversion_schedule.append(conversion)
        year_data['conversion'] = conversion
        year_data['conv_tax'] = conversion * 0.24 if conversion > 0 else 0
        
        # === INCOME NEEDS ===
        income_need = ANNUAL_INCOME_NEED * (1 + INFLATION_RATE) ** yr
        ss1 = SPOUSE1_SS_ANNUAL if yr >= years_to_rajesh_ss else 0
        ss2 = SPOUSE2_SS_ANNUAL if yr >= years_to_terri_ss else 0
        total_ss = ss1 + ss2
        
        # RMDs count as income (can cover some needs)
        from_rmd_for_spending = min(total_rmd_this_year * 0.76, income_need - total_ss)  # After tax
        remaining_need = max(0, income_need - total_ss - from_rmd_for_spending)
        
        # Draw from accounts for remaining need
        from_taxable = min(remaining_need * 0.3, taxable - MINIMUM_CASH_RESERVE)
        from_roth = min(remaining_need * 0.2, roth)
        from_ira = remaining_need - from_taxable - from_roth
        
        taxable -= max(0, from_taxable)
        roth -= max(0, from_roth)
        ira -= max(0, from_ira * 1.24)  # Gross up for tax
        
        year_data['income_need'] = income_need
        year_data['ss_income'] = total_ss
        
        # === GROWTH ===
        ira *= (1 + IRA_RETURN)
        roth *= (1 + ROTH_RETURN)
        taxable *= (1 + TAXABLE_RETURN)
        
        # === END OF YEAR ===
        year_data['ira_end'] = ira
        year_data['roth_end'] = roth
        year_data['taxable_end'] = taxable
        
        yearly_data.append(year_data)
    
    # Final calculations
    after_tax = ira * 0.75 + roth + taxable * 0.92
    legacy = ira * (1 - 0.28) + roth + taxable * 0.95
    
    return {
        'total_conversions': total_conversions,
        'total_rmds': total_rmds,
        'total_rmd_tax': total_rmd_tax,
        'conversion_schedule': conversion_schedule,
        'rmd_schedule': rmd_schedule,
        'yearly_data': yearly_data,
        'ira': ira,
        'roth': roth,
        'taxable': taxable,
        'after_tax': after_tax,
        'legacy': legacy
    }

# =============================================================================
# RUN THE ANALYSIS
# =============================================================================

print("═" * 80)
print(f"🏠 CHAPTER 8: HOME PURCHASE ANALYSIS (${HOME_DOWN_PAYMENT:,} DOWN PAYMENT)")
print(f"   📅 Planned Purchase: {HOME_PURCHASE_YEAR} (Year {PURCHASE_YEAR_OFFSET + 1} of retirement)")
print(f"   📊 NOW INCLUDES RMD CALCULATIONS!")
print("═" * 80)

# Run scenarios
no_home = project_with_rmds(99, 0, TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE_ACCOUNTS)
buy_2025 = project_with_rmds(0, HOME_DOWN_PAYMENT, TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE_ACCOUNTS)
buy_2026 = project_with_rmds(1, HOME_DOWN_PAYMENT, TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE_ACCOUNTS)
buy_2027 = project_with_rmds(2, HOME_DOWN_PAYMENT, TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE_ACCOUNTS)
selected = project_with_rmds(PURCHASE_YEAR_OFFSET, HOME_DOWN_PAYMENT, TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE_ACCOUNTS)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         YOUR SITUATION                                      │
└─────────────────────────────────────────────────────────────────────────────┘

  💵 Down Payment Needed:    ${HOME_DOWN_PAYMENT:>12,}
  📅 Planned Purchase Year:  {HOME_PURCHASE_YEAR}
  
  📊 Current Assets:
     ├── Taxable Account:    ${JOINT_TAXABLE_ACCOUNTS:>12,}
     ├── IRAs (pre-tax):     ${TOTAL_PRETAX:>12,}
     └── Roth IRAs:          ${TOTAL_ROTH:>12,}

  ⏰ RMD Timeline:
     ├── {SPOUSE2_NAME} turns 73:     {2025 + years_to_terri_rmd - 1} - RMDs begin
     └── {SPOUSE1_NAME} turns 73:    {2025 + years_to_rajesh_rmd - 1} - RMDs begin
""")

# RMD ANALYSIS
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│              📊 RMD IMPACT: WHY THIS MATTERS FOR HOME PURCHASE              │
└─────────────────────────────────────────────────────────────────────────────┘
""")

print(f"""
  🔴 WITHOUT HOME PURCHASE (Maximum Conversions):
     • Total Roth Conversions:  ${no_home['total_conversions']:>12,.0f}
     • Total RMDs over 25 yrs:  ${no_home['total_rmds']:>12,.0f}
     • RMD Tax Paid:            ${no_home['total_rmd_tax']:>12,.0f}
  
  🏠 WITH HOME PURCHASE IN {HOME_PURCHASE_YEAR}:
     • Total Roth Conversions:  ${selected['total_conversions']:>12,.0f}  ({selected['total_conversions'] - no_home['total_conversions']:+,.0f})
     • Total RMDs over 25 yrs:  ${selected['total_rmds']:>12,.0f}  ({selected['total_rmds'] - no_home['total_rmds']:+,.0f})
     • RMD Tax Paid:            ${selected['total_rmd_tax']:>12,.0f}  ({selected['total_rmd_tax'] - no_home['total_rmd_tax']:+,.0f})

  💡 KEY INSIGHT: 
     Fewer conversions → Larger IRA → Higher RMDs → More forced income → More tax
""")

# TIMING COMPARISON
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│              📅 TIMING COMPARISON: WHEN TO BUY THE HOME                     │
└─────────────────────────────────────────────────────────────────────────────┘
""")

print(f"""
╔═════════════════════════╦══════════════╦══════════════╦══════════════╦══════════════╗
║                         ║   NO HOME    ║  BUY 2025    ║  BUY 2026    ║  BUY 2027    ║
║       METRIC            ║  (Baseline)  ║   (Year 1)   ║   (Year 2)   ║   (Year 3)   ║
╠═════════════════════════╬══════════════╬══════════════╬══════════════╬══════════════╣
║ Roth Conversions        ║ ${no_home['total_conversions']:>10,.0f}  ║ ${buy_2025['total_conversions']:>10,.0f}  ║ ${buy_2026['total_conversions']:>10,.0f}  ║ ${buy_2027['total_conversions']:>10,.0f}  ║
║ Total RMDs (25 yrs)     ║ ${no_home['total_rmds']:>10,.0f}  ║ ${buy_2025['total_rmds']:>10,.0f}  ║ ${buy_2026['total_rmds']:>10,.0f}  ║ ${buy_2027['total_rmds']:>10,.0f}  ║
║ RMD Tax Paid            ║ ${no_home['total_rmd_tax']:>10,.0f}  ║ ${buy_2025['total_rmd_tax']:>10,.0f}  ║ ${buy_2026['total_rmd_tax']:>10,.0f}  ║ ${buy_2027['total_rmd_tax']:>10,.0f}  ║
╠═════════════════════════╬══════════════╬══════════════╬══════════════╬══════════════╣
║ 25-Year Wealth          ║ ${no_home['after_tax']:>10,.0f}  ║ ${buy_2025['after_tax']:>10,.0f}  ║ ${buy_2026['after_tax']:>10,.0f}  ║ ${buy_2027['after_tax']:>10,.0f}  ║
║ Legacy to Kids          ║ ${no_home['legacy']:>10,.0f}  ║ ${buy_2025['legacy']:>10,.0f}  ║ ${buy_2026['legacy']:>10,.0f}  ║ ${buy_2027['legacy']:>10,.0f}  ║
╠═════════════════════════╬══════════════╬══════════════╬══════════════╬══════════════╣
║ vs No Home              ║ $         0  ║ ${buy_2025['after_tax'] - no_home['after_tax']:>+10,.0f}  ║ ${buy_2026['after_tax'] - no_home['after_tax']:>+10,.0f}  ║ ${buy_2027['after_tax'] - no_home['after_tax']:>+10,.0f}  ║
║ vs Buy 2025             ║      ---     ║ $         0  ║ ${buy_2026['after_tax'] - buy_2025['after_tax']:>+10,.0f}  ║ ${buy_2027['after_tax'] - buy_2025['after_tax']:>+10,.0f}  ║
╚═════════════════════════╩══════════════╩══════════════╩══════════════╩══════════════╝
""")

# YEAR BY YEAR FOR SELECTED TIMING
print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│    📅 YOUR PLAN: BUY IN {HOME_PURCHASE_YEAR} - YEAR-BY-YEAR BREAKDOWN{'':>26}│
└─────────────────────────────────────────────────────────────────────────────┘
""")

print("  Year  Calendar  Ages    Conversion    RMD         Notes")
print("  ────  ────────  ─────   ──────────    ──────────  ─────────────────────")
for i, yd in enumerate(selected['yearly_data'][:15]):  # Show first 15 years
    ages = f"{yd['rajesh_age']}/{yd['terri_age']}"
    conv = f"${yd['conversion']:>9,.0f}" if yd['conversion'] > 0 else "         -"
    rmd = f"${yd['rmd']:>9,.0f}" if yd['rmd'] > 0 else "         -"
    
    notes = []
    if yd.get('home_purchase'):
        notes.append("🏠 BUY HOME")
    if yd['terri_age'] == 73:
        notes.append(f"{SPOUSE2_NAME} RMD starts")
    if yd['rajesh_age'] == 73:
        notes.append(f"{SPOUSE1_NAME} RMD starts")
    if i + 1 == years_to_terri_ss:
        notes.append(f"{SPOUSE2_NAME} SS +${SPOUSE2_SS_ANNUAL:,}")
    if i + 1 == years_to_rajesh_ss:
        notes.append(f"{SPOUSE1_NAME} SS +${SPOUSE1_SS_ANNUAL:,}")
    
    note_str = ", ".join(notes) if notes else ""
    print(f"  {yd['year']:>4}  {yd['calendar_year']:>8}  {ages:<6}  {conv}    {rmd}  {note_str}")

print(f"""
  ... (Years 16-25 continue with RMDs and growth)

  ═══════════════════════════════════════════════════════════════════════════
  TOTALS (25 years):
     • Total Conversions: ${selected['total_conversions']:>12,.0f}
     • Total RMDs:        ${selected['total_rmds']:>12,.0f}
     • Total RMD Tax:     ${selected['total_rmd_tax']:>12,.0f}
  ═══════════════════════════════════════════════════════════════════════════
""")

# RECOMMENDATION
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                       🎯 RECOMMENDATION                                     │
└─────────────────────────────────────────────────────────────────────────────┘
""")

best_option = max([(buy_2025, 2025), (buy_2026, 2026), (buy_2027, 2027)], key=lambda x: x[0]['after_tax'])
best_result, best_year = best_option
benefit_vs_2025 = best_result['after_tax'] - buy_2025['after_tax']

print(f"""
  📊 ANALYSIS SUMMARY (with ${HOME_DOWN_PAYMENT:,} down payment):

  ✅ BEST TIMING: Buy in {best_year}
     • 25-Year Wealth:        ${best_result['after_tax']:>12,.0f}
     • Legacy to Kids:        ${best_result['legacy']:>12,.0f}
     • Total Roth Conversions:${best_result['total_conversions']:>12,.0f}
     • Total RMDs:            ${best_result['total_rmds']:>12,.0f}
""")

if best_year > 2025:
    print(f"""
  💰 VALUE OF WAITING TO {best_year}:
     • Extra wealth vs 2025:   ${benefit_vs_2025:>+12,.0f}
     • Extra conversions:      ${best_result['total_conversions'] - buy_2025['total_conversions']:>+12,.0f}
     • Lower RMDs:             ${best_result['total_rmds'] - buy_2025['total_rmds']:>+12,.0f}
     
  WHY? Delaying allows more Roth conversions BEFORE the down payment
  depletes your taxable account. More conversions = lower IRA = lower RMDs
  = less forced taxable income later.
""")

# Cost of home vs no home
cost_of_home = no_home['after_tax'] - best_result['after_tax']
print(f"""
  🏠 TOTAL COST OF BUYING THE HOME:
     • Wealth reduction vs no home: ${cost_of_home:>12,.0f}
     • That's ${cost_of_home/25:,.0f}/year in lost wealth accumulation
     • BUT: You get a home! Quality of life has value too.

  ⚠️  LIFE CONSIDERATIONS:
     • Can you wait until {best_year} to buy?
     • Are home prices rising faster than your gains from waiting?
     • What's the rental/housing cost while waiting?
     • Is there a perfect home available NOW?

  🔧 TO EXPLORE: Change HOME_DOWN_PAYMENT (amount) and HOME_PURCHASE_YEAR 
     (2025, 2026, 2027, etc.) at the top of this cell and re-run.
""")

In [ ]:
# =============================================================================
# 📊 CHAPTER 9: THE 32% QUESTION
# =============================================================================
# 
# THE QUESTION: If I convert aggressively and pay 32% tax now, will the 
# savings from lower RMDs later make up for it? And when does that happen?
#
# This requires comparing:
# - Scenario A: Stay in 24% bracket (conservative conversions)
# - Scenario B: Push into 32% bracket (aggressive conversions)
#
# We need to track cumulative tax paid over time and find the crossover point.
# =============================================================================

print("═" * 80)
print("📊 CHAPTER 9: THE 32% QUESTION - IS AGGRESSIVE CONVERTING WORTH IT?")
print("═" * 80)

# =============================================================================
# ENHANCED PROJECTION WITH CUMULATIVE TAX TRACKING
# =============================================================================

def project_with_tax_tracking_ch9(annual_conversion, conversion_years, allow_32_bracket=False):
    """
    Project with detailed tax tracking to find breakeven points.
    
    Parameters:
    -----------
    annual_conversion : float
        Target annual conversion amount
    conversion_years : int
        How many years to do conversions
    allow_32_bracket : bool
        If True, allow conversions that push into 32% bracket
        If False, cap conversions at 24% bracket ceiling
    """
    
    # Starting balances
    ira = TOTAL_PRETAX
    roth = TOTAL_ROTH
    taxable = JOINT_TAXABLE_ACCOUNTS
    
    # Tracking
    yearly_data = []
    cumulative_conv_tax = 0
    cumulative_rmd_tax = 0
    cumulative_total_tax = 0
    total_conversions = 0
    total_rmds = 0
    
    for yr in range(25):
        rajesh_age = SPOUSE1_AGE + yr
        terri_age = SPOUSE2_AGE + yr
        
        # === INCOME SOURCES ===
        ss1 = SPOUSE1_SS_ANNUAL if yr >= years_to_rajesh_ss else 0
        ss2 = SPOUSE2_SS_ANNUAL if yr >= years_to_terri_ss else 0
        total_ss = ss1 + ss2
        ss_taxable = total_ss * 0.85
        
        income_need = ANNUAL_INCOME_NEED * (1 + INFLATION_RATE) ** yr
        from_savings_needed = max(0, income_need - total_ss)
        
        # === RMDs ===
        rajesh_rmd = calculate_rmd(ira * 0.33, rajesh_age)
        terri_rmd = calculate_rmd(ira * 0.67, terri_age)
        total_rmd = rajesh_rmd + terri_rmd
        total_rmds += total_rmd
        
        rmd_for_income = min(total_rmd, from_savings_needed)
        remaining_need = from_savings_needed - rmd_for_income
        
        # === WITHDRAWALS ===
        from_taxable = min(remaining_need * 0.5, max(0, taxable - MINIMUM_CASH_RESERVE))
        from_roth = min(remaining_need * 0.3, roth)
        from_ira_extra = max(0, remaining_need - from_taxable - from_roth)
        total_ira_withdrawal = total_rmd + from_ira_extra
        
        # === BASE TAXABLE INCOME (before conversions) ===
        base_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
        base_taxable_income = max(0, base_taxable_income)
        
        # === ROTH CONVERSIONS ===
        conversion = 0
        conversion_tax = 0
        conversion_bracket = 0
        
        if yr < conversion_years and annual_conversion > 0:
            available_for_conv_tax = max(0, taxable - MINIMUM_CASH_RESERVE - from_taxable)
            
            # Bracket limits (MFJ 2024)
            bracket_24_ceiling = 383900  # Top of 24% bracket
            bracket_32_ceiling = 487450  # Top of 32% bracket
            
            if allow_32_bracket:
                # Allow conversions up to 32% bracket ceiling
                room_in_brackets = max(0, bracket_32_ceiling - base_taxable_income)
            else:
                # Stay within 24% bracket
                room_in_brackets = max(0, bracket_24_ceiling - base_taxable_income)
            
            max_affordable = available_for_conv_tax / 0.28 if available_for_conv_tax > 0 else 0
            conversion = min(annual_conversion, room_in_brackets, max_affordable, ira - total_ira_withdrawal)
            conversion = max(0, conversion)
            
            if conversion > 0:
                income_after_conv = base_taxable_income + conversion
                conversion_tax = calculate_tax_mfj(income_after_conv) - calculate_tax_mfj(base_taxable_income)
                
                # Determine what bracket the conversion hit
                if base_taxable_income + conversion > bracket_24_ceiling:
                    conversion_bracket = 32
                elif base_taxable_income + conversion > 201050:
                    conversion_bracket = 24
                else:
                    conversion_bracket = 22
                
                total_conversions += conversion
                cumulative_conv_tax += conversion_tax
        
        # === RMD TAX ===
        total_taxable_income = ss_taxable + total_ira_withdrawal - STANDARD_DEDUCTION_MFJ
        total_taxable_income = max(0, total_taxable_income)
        income_tax = calculate_tax_mfj(total_taxable_income)
        
        if total_ira_withdrawal > 0:
            rmd_share = total_rmd / total_ira_withdrawal
            rmd_tax = income_tax * rmd_share
        else:
            rmd_tax = 0
        
        cumulative_rmd_tax += rmd_tax
        cumulative_total_tax = cumulative_conv_tax + cumulative_rmd_tax
        
        # === UPDATE BALANCES ===
        ira -= total_ira_withdrawal
        ira -= conversion
        roth += conversion
        roth -= from_roth
        taxable -= from_taxable
        taxable -= conversion_tax
        taxable -= income_tax * 0.5
        
        ira *= (1 + IRA_RETURN)
        roth *= (1 + ROTH_RETURN)
        taxable *= (1 + TAXABLE_RETURN)
        
        ira = max(0, ira)
        roth = max(0, roth)
        taxable = max(0, taxable)
        
        yearly_data.append({
            'year': yr + 1,
            'calendar_year': 2025 + yr,
            'rajesh_age': rajesh_age,
            'terri_age': terri_age,
            'conversion': conversion,
            'conversion_tax': conversion_tax,
            'conversion_bracket': conversion_bracket,
            'rmd': total_rmd,
            'rmd_tax': rmd_tax,
            'cumulative_conv_tax': cumulative_conv_tax,
            'cumulative_rmd_tax': cumulative_rmd_tax,
            'cumulative_total_tax': cumulative_total_tax,
            'ira': ira,
            'roth': roth,
            'taxable': taxable
        })
    
    after_tax = ira * 0.75 + roth + taxable * 0.92
    legacy = ira * (1 - 0.28) + roth + taxable * 0.95
    
    return {
        'total_conversions': total_conversions,
        'total_conv_tax': cumulative_conv_tax,
        'total_rmds': total_rmds,
        'total_rmd_tax': cumulative_rmd_tax,
        'total_lifetime_tax': cumulative_total_tax,
        'after_tax': after_tax,
        'legacy': legacy,
        'yearly_data': yearly_data
    }

# =============================================================================
# RUN THE SCENARIOS
# =============================================================================

# Scenario 1: Conservative - Stay in 24% bracket ($100K/yr for 5 years)
conservative = project_with_tax_tracking_ch9(100_000, 5, allow_32_bracket=False)

# Scenario 2: Aggressive - Push into 32% bracket ($175K/yr for 8 years)
aggressive = project_with_tax_tracking_ch9(175_000, 8, allow_32_bracket=True)

# Scenario 3: Very Aggressive - Max conversions ($200K/yr for 10 years)
very_aggressive = project_with_tax_tracking_ch9(200_000, 10, allow_32_bracket=True)

# Scenario 4: Do Nothing (baseline)
do_nothing = project_with_tax_tracking_ch9(0, 0, allow_32_bracket=False)

# =============================================================================
# DISPLAY THE ANALYSIS
# =============================================================================

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    THE QUESTION YOU'RE REALLY ASKING                        │
└─────────────────────────────────────────────────────────────────────────────┘

  "If I pay 32% tax on conversions NOW, will the reduced RMDs LATER
   save me enough to make it worthwhile? And when does breakeven occur?"

  This is the right question! Let's analyze it.

┌─────────────────────────────────────────────────────────────────────────────┐
│                    THE SCENARIOS                                            │
└─────────────────────────────────────────────────────────────────────────────┘

  📊 SCENARIO COMPARISON:
  
  ┌──────────────────────┬─────────────────┬─────────────────┬─────────────────┐
  │                      │  CONSERVATIVE   │   AGGRESSIVE    │ VERY AGGRESSIVE │
  │                      │  (Stay ≤24%)    │  (Allow 32%)    │   (Max 32%)     │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ Conversion Target    │ $100K/yr × 5yr  │ $175K/yr × 8yr  │ $200K/yr × 10yr │
  │ Total Converted      │ ${conservative['total_conversions']:>13,.0f} │ ${aggressive['total_conversions']:>13,.0f} │ ${very_aggressive['total_conversions']:>13,.0f} │
  │ Conversion Tax Paid  │ ${conservative['total_conv_tax']:>13,.0f} │ ${aggressive['total_conv_tax']:>13,.0f} │ ${very_aggressive['total_conv_tax']:>13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ Total RMDs (25 yrs)  │ ${conservative['total_rmds']:>13,.0f} │ ${aggressive['total_rmds']:>13,.0f} │ ${very_aggressive['total_rmds']:>13,.0f} │
  │ Total RMD Tax        │ ${conservative['total_rmd_tax']:>13,.0f} │ ${aggressive['total_rmd_tax']:>13,.0f} │ ${very_aggressive['total_rmd_tax']:>13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ LIFETIME TAX         │ ${conservative['total_lifetime_tax']:>13,.0f} │ ${aggressive['total_lifetime_tax']:>13,.0f} │ ${very_aggressive['total_lifetime_tax']:>13,.0f} │
  │ vs Conservative      │ ${0:>13,.0f} │ ${aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:>+13,.0f} │ ${very_aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:>+13,.0f} │
  ├──────────────────────┼─────────────────┼─────────────────┼─────────────────┤
  │ After-Tax Wealth     │ ${conservative['after_tax']:>13,.0f} │ ${aggressive['after_tax']:>13,.0f} │ ${very_aggressive['after_tax']:>13,.0f} │
  │ Legacy to Kids       │ ${conservative['legacy']:>13,.0f} │ ${aggressive['legacy']:>13,.0f} │ ${very_aggressive['legacy']:>13,.0f} │
  └──────────────────────┴─────────────────┴─────────────────┴─────────────────┘
""")

# =============================================================================
# BREAKEVEN ANALYSIS
# =============================================================================

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│              📈 BREAKEVEN ANALYSIS: WHEN DOES 32% PAY OFF?                  │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Find the crossover point
print("  Year  Conservative         Aggressive           Difference")
print("        Cumul. Tax           Cumul. Tax           (Neg = Aggressive ahead)")
print("  ────  ───────────────────  ───────────────────  ─────────────────────")

crossover_year = None
for i in range(25):
    cons_data = conservative['yearly_data'][i]
    agg_data = aggressive['yearly_data'][i]
    
    diff = agg_data['cumulative_total_tax'] - cons_data['cumulative_total_tax']
    
    if diff < 0 and crossover_year is None:
        crossover_year = i + 1
        marker = " ← BREAKEVEN!"
    elif diff < 0:
        marker = " ✓"
    else:
        marker = ""
    
    # Show key years
    if i < 10 or i >= 11 or (crossover_year and abs(i + 1 - crossover_year) <= 1):
        print(f"  {i+1:>4}  ${cons_data['cumulative_total_tax']:>17,.0f}  ${agg_data['cumulative_total_tax']:>17,.0f}  ${diff:>+18,.0f}{marker}")

print(f"""
  ═══════════════════════════════════════════════════════════════════════════
""")

# =============================================================================
# THE VERDICT
# =============================================================================

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                       🎯 THE VERDICT                                        │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Determine best strategy
best_wealth = max(conservative['after_tax'], aggressive['after_tax'], very_aggressive['after_tax'])
if best_wealth == very_aggressive['after_tax']:
    best_strategy = "VERY AGGRESSIVE (200K/yr, allow 32%)"
    best = very_aggressive
elif best_wealth == aggressive['after_tax']:
    best_strategy = "AGGRESSIVE (175K/yr, allow 32%)"
    best = aggressive
else:
    best_strategy = "CONSERVATIVE (100K/yr, stay ≤24%)"
    best = conservative

lowest_tax = min(conservative['total_lifetime_tax'], aggressive['total_lifetime_tax'], very_aggressive['total_lifetime_tax'])
if lowest_tax == very_aggressive['total_lifetime_tax']:
    lowest_tax_strategy = "VERY AGGRESSIVE"
elif lowest_tax == aggressive['total_lifetime_tax']:
    lowest_tax_strategy = "AGGRESSIVE"
else:
    lowest_tax_strategy = "CONSERVATIVE"

print(f"""
  📊 WHAT THE NUMBERS SHOW:

  ┌─────────────────────────────────────────────────────────────────────────┐
  │ LOWEST LIFETIME TAX:    {lowest_tax_strategy:>20}                       │
  │ HIGHEST AFTER-TAX WEALTH: {best_strategy:>45} │
  └─────────────────────────────────────────────────────────────────────────┘
""")

if crossover_year:
    print(f"""
  ⏱️  BREAKEVEN TIMELINE:
     
     Aggressive strategy breaks even in YEAR {crossover_year} ({2024 + crossover_year})
     That's when {SPOUSE1_NAME} is {SPOUSE1_AGE + crossover_year - 1} and {SPOUSE2_NAME} is {SPOUSE2_AGE + crossover_year - 1}
     
     After Year {crossover_year}, every additional year FAVORS the aggressive strategy
     because the reduced RMDs keep compounding the savings.
""")
else:
    tax_diff = aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']
    print(f"""
  ⚠️  BREAKEVEN NOT REACHED IN 25 YEARS
     
     The aggressive strategy pays ${tax_diff:,.0f} MORE in lifetime taxes
     The extra conversion tax is NOT recovered through lower RMDs
     in the 25-year projection window.
""")

wealth_diff = best['after_tax'] - conservative['after_tax']
legacy_diff = best['legacy'] - conservative['legacy']

print(f"""
  💰 WEALTH COMPARISON (Aggressive vs Conservative):
     • After-Tax Wealth: ${aggressive['after_tax'] - conservative['after_tax']:>+15,.0f}
     • Legacy to Kids:   ${aggressive['legacy'] - conservative['legacy']:>+15,.0f}
     • Lifetime Tax:     ${aggressive['total_lifetime_tax'] - conservative['total_lifetime_tax']:>+15,.0f}

  🤔 THE TRADEOFF:
""")

if aggressive['after_tax'] > conservative['after_tax']:
    print(f"""
     ✅ YES, paying 32% NOW is worth it!
     
     Even though you pay higher tax rates upfront, the benefits compound:
     • Lower RMDs = less forced taxable income
     • More in Roth = tax-free growth for decades
     • Better legacy = kids inherit tax-free dollars
     
     The math shows aggressive conversion wins by ${aggressive['after_tax'] - conservative['after_tax']:,.0f}
     in after-tax wealth over 25 years.
""")
else:
    print(f"""
     ❌ NO, staying in 24% is the better choice
     
     The 32% tax rate is too steep a price to pay:
     • The extra tax upfront isn't recovered fast enough
     • Conservative conversion is more efficient
     • You keep more of your money
     
     Conservative wins by ${conservative['after_tax'] - aggressive['after_tax']:,.0f}
     in after-tax wealth over 25 years.
""")

print(f"""
  ════════════════════════════════════════════════════════════════════════════

  🔑 KEY INSIGHT:
  
     The answer depends on YOUR timeline and goals:
     
     • If you expect to live 20+ years past RMD start → Aggressive may win
     • If legacy to kids matters most → Aggressive likely wins (Roth is tax-free)
     • If minimizing lifetime tax is the goal → Check which strategy wins above
     • If you need flexibility/liquidity → Conservative preserves taxable account

  ════════════════════════════════════════════════════════════════════════════
""")

# 🔎 What “Conservative vs Aggressive” Means Here (and why it matters)

In this notebook, **“Conservative / Aggressive / Very Aggressive” refers to the *Roth conversion plan*** — *not* whether you invest conservatively or aggressively.

## 1) The three conversion strategies (the thing we’re comparing)
All three strategies use the same retirement spending/RMD logic; they only differ in how hard you convert IRA → Roth in the early years.

- **Conservative conversion strategy**
  - Convert **$100K per year for 5 years**
  - **Cap conversions at the 24% bracket ceiling** (do *not* intentionally convert into 32%)
  - Intuition: reduce future RMDs while avoiding higher marginal tax rates today.

- **Aggressive conversion strategy**
  - Convert **$175K per year for 8 years**
  - **Allow conversions into the 32% bracket** when needed
  - Intuition: pay more tax earlier to move more dollars into Roth sooner, reducing later taxable income (RMDs) and increasing tax-free compounding.

- **Very Aggressive conversion strategy**
  - Convert **$200K per year for 10 years**
  - **Allow 32% bracket**
  - Intuition: push hardest on lowering future RMDs and maximizing Roth balances (at the cost of higher taxes paid earlier).

## 2) The investment model is separate from the conversion strategy
The Monte Carlo section models market uncertainty via **stocks + bonds (with correlation)** and applies different allocations per account (example defaults):
- IRA: **60/40** stocks/bonds
- Roth: **80/20**
- Taxable: **70/30** (plus a small “tax drag” simplification)

So: **each conversion strategy is run under the same simulated stock/bond market paths** (apples-to-apples). Differences in outcomes come from **tax timing + account location (IRA vs Roth vs taxable)**, not “getting luckier returns.”

## 3) Why it’s important to consider multiple conversion strategies
- **Tax-rate tradeoff:** converting more now can reduce future RMD-driven taxable income, but may push you into higher brackets today.
- **Liquidity/cash tradeoff:** conversion taxes are paid from taxable/cash; converting too much can reduce near-term flexibility.
- **Downside protection:** in poor market paths, the “best” conversion intensity can change because early taxes paid may not be recovered by later tax savings.
- **Legacy & flexibility:** higher Roth balances can improve tax flexibility later (and often improves after-tax legacy), but the benefit depends on the path and assumptions.

The Monte Carlo results help answer: **How often does Aggressive beat Conservative, and by how much, across many market/inflation paths?**

In [ ]:
# =============================================================================
# 📉 Monte Carlo Analysis (B): Stocks/Bonds + correlation + account allocations
# =============================================================================
# Approach B inputs add:
#   - Stock + bond return distributions (mean/std)
#   - Stock/bond correlation
#   - Account allocation weights (stock %) for IRA/Roth/Taxable
#   - Optional taxable drag (tax friction)
# =============================================================================

print("═" * 80)
print("📉 MONTE CARLO (B): STOCKS/BONDS + CORRELATION + ACCOUNT ALLOCATIONS")
print("═" * 80)

# ----------------------------
# 1) Monte Carlo INPUTS (edit these)
# ----------------------------
MC_N_SIMS = 5_000
MC_HORIZON_YEARS = 25
MC_SEED = 42

# Asset return assumptions (annual, nominal)
MC_STOCK_MEAN = 0.08
MC_STOCK_STD = 0.18
MC_BOND_MEAN = 0.04
MC_BOND_STD = 0.07
MC_STOCK_BOND_CORR = 0.10  # typical low positive / near-zero correlation

# Inflation assumptions
MC_INFL_MEAN = INFLATION_RATE
MC_INFL_STD = 0.01

# Account allocations (stock weight). These match your Chapter 1 comments.
IRA_STOCK_WT = 0.60
ROTH_STOCK_WT = 0.80
TAXABLE_STOCK_WT = 0.70

# Optional tax drag on taxable returns (very simplified).
# Example: 0.50% means subtract 0.005 from the taxable portfolio return each year.
MC_TAXABLE_DRAG = 0.005

# Safety clamps
MC_ASSET_RETURN_MIN = -0.95
MC_ASSET_RETURN_MAX = 1.00
MC_INFL_MIN = -0.02
MC_INFL_MAX = 0.20

# Strategies to run (all 3)
MC_STRATEGIES = [
    dict(name="Conservative (100K x 5, ≤24%)", annual_conversion=100_000, conversion_years=5, allow_32_bracket=False),
    dict(name="Aggressive (175K x 8, allow 32%)", annual_conversion=175_000, conversion_years=8, allow_32_bracket=True),
    dict(name="Very Aggressive (200K x 10, allow 32%)", annual_conversion=200_000, conversion_years=10, allow_32_bracket=True),
 ]

# ----------------------------
# 2) Simulation helpers (apples-to-apples paths)
# ----------------------------
def _simulate_asset_paths_B(*, n_sims: int, horizon_years: int, seed: Optional[int]):
    rng = np.random.default_rng(seed)
    # Correlated normals for stock/bond returns
    corr = float(MC_STOCK_BOND_CORR)
    corr = max(-0.99, min(0.99, corr))
    cov = np.array([[MC_STOCK_STD**2, corr * MC_STOCK_STD * MC_BOND_STD],
                    [corr * MC_STOCK_STD * MC_BOND_STD, MC_BOND_STD**2]], dtype=float)
    L = np.linalg.cholesky(cov)
    z = rng.normal(size=(n_sims, horizon_years, 2))
    shocks = z @ L.T
    stock = MC_STOCK_MEAN + shocks[..., 0]
    bond = MC_BOND_MEAN + shocks[..., 1]
    stock = np.clip(stock, MC_ASSET_RETURN_MIN, MC_ASSET_RETURN_MAX)
    bond = np.clip(bond, MC_ASSET_RETURN_MIN, MC_ASSET_RETURN_MAX)

    infl = rng.normal(loc=MC_INFL_MEAN, scale=MC_INFL_STD, size=(n_sims, horizon_years))
    infl = np.clip(infl, MC_INFL_MIN, MC_INFL_MAX)
    return stock, bond, infl

def _portfolio_return(stock_r: np.ndarray, bond_r: np.ndarray, stock_wt: float) -> np.ndarray:
    w = float(stock_wt)
    w = max(0.0, min(1.0, w))
    return w * stock_r + (1.0 - w) * bond_r

def run_monte_carlo_B(*, strategy: dict, stock: np.ndarray, bond: np.ndarray, infl: np.ndarray, horizon_years: int):
    rows = []
    for sim in range(stock.shape[0]):
        stock_r = stock[sim]
        bond_r = bond[sim]
        infl_r = infl[sim]

        ira_r = _portfolio_return(stock_r, bond_r, IRA_STOCK_WT)
        roth_r = _portfolio_return(stock_r, bond_r, ROTH_STOCK_WT)
        taxable_r = _portfolio_return(stock_r, bond_r, TAXABLE_STOCK_WT) - MC_TAXABLE_DRAG

        out = project_with_tax_tracking(
            annual_conversion=strategy["annual_conversion"],
            conversion_years=strategy["conversion_years"],
            allow_32_bracket=strategy["allow_32_bracket"],
            total_pretax=TOTAL_PRETAX,
            total_roth=TOTAL_ROTH,
            joint_taxable=JOINT_TAXABLE_ACCOUNTS,
            spouse1_age=SPOUSE1_AGE,
            spouse2_age=SPOUSE2_AGE,
            spouse1_ss=SPOUSE1_SS_ANNUAL,
            spouse2_ss=SPOUSE2_SS_ANNUAL,
            years_to_s1_ss=years_to_rajesh_ss,
            years_to_s2_ss=years_to_terri_ss,
            annual_income=ANNUAL_INCOME_NEED,
            min_cash=MINIMUM_CASH_RESERVE,
            # Scalars for compatibility; series override them
            ira_return=IRA_RETURN,
            roth_return=ROTH_RETURN,
            taxable_return=TAXABLE_RETURN,
            inflation=INFLATION_RATE,
            horizon_years=horizon_years,
            ira_returns=ira_r,
            roth_returns=roth_r,
            taxable_returns=taxable_r,
            inflation_rates=infl_r,
        )

        path = out["yearly_data"]
        totals = np.array([d["ira"] + d["roth"] + d["taxable"] for d in path], dtype=float)
        min_total = float(np.min(totals))
        end_total = float(totals[-1])
        success = min_total > 0.0

        rows.append({
            "strategy": strategy["name"],
            "sim": sim,
            "after_tax": out["after_tax"],
            "legacy": out["legacy"],
            "total_lifetime_tax": out["total_lifetime_tax"],
            "end_total_balance": end_total,
            "min_total_balance": min_total,
            "success": success,
        })
    return pd.DataFrame(rows)

# ----------------------------
# 3) Cleaner summaries
# ----------------------------
def _percentile_table(df: pd.DataFrame, metrics: List[str], ps=(0.05, 0.50, 0.95)) -> pd.DataFrame:
    out = {}
    for m in metrics:
        s = df[m].astype(float)
        out[m] = {f"p{int(p*100)}": float(s.quantile(p)) for p in ps}
    tab = pd.DataFrame(out).T
    tab.index.name = "metric"
    return tab

def _fmt_money(x: float) -> str:
    if np.isnan(x):
        return ""
    ax = abs(x)
    if ax >= 1_000_000:
        return f"${x/1_000_000:,.2f}M"
    return f"${x:,.0f}"

def _print_strategy_summary(label: str, df: pd.DataFrame):
    sr = float(df["success"].mean())
    metrics = ["after_tax", "legacy", "end_total_balance", "min_total_balance", "total_lifetime_tax"]
    pt = _percentile_table(df, metrics, ps=(0.05, 0.25, 0.50, 0.75, 0.95))
    pt = pt.applymap(_fmt_money)
    print(f"\n{label}")
    print(f"Success rate (never hit $0 total): {sr:.1%}")
    display(pt)

# ----------------------------
# 4) Run all strategies under the same simulated market paths
# ----------------------------
stock, bond, infl = _simulate_asset_paths_B(
    n_sims=MC_N_SIMS,
    horizon_years=MC_HORIZON_YEARS,
    seed=MC_SEED,
 )

results = []
for strat in MC_STRATEGIES:
    results.append(run_monte_carlo_B(strategy=strat, stock=stock, bond=bond, infl=infl, horizon_years=MC_HORIZON_YEARS))

mc_all = pd.concat(results, ignore_index=True)

print(f"Simulations per strategy: {MC_N_SIMS:,} | Horizon: {MC_HORIZON_YEARS} years")
print(f"Stocks: mean={MC_STOCK_MEAN:.2%}, std={MC_STOCK_STD:.2%} | Bonds: mean={MC_BOND_MEAN:.2%}, std={MC_BOND_STD:.2%} | corr={MC_STOCK_BOND_CORR:.2f}")
print(f"Inflation: mean={MC_INFL_MEAN:.2%}, std={MC_INFL_STD:.2%} | Taxable drag: {MC_TAXABLE_DRAG:.2%}")
print(f"Allocations (stock %): IRA={IRA_STOCK_WT:.0%}, Roth={ROTH_STOCK_WT:.0%}, Taxable={TAXABLE_STOCK_WT:.0%}")

# Compact comparison
comparison_rows = []
for strat in MC_STRATEGIES:
    name = strat["name"]
    df = mc_all[mc_all["strategy"] == name]
    comparison_rows.append({
        "strategy": name,
        "success_rate": float(df["success"].mean()),
        "after_tax_p50": float(df["after_tax"].quantile(0.50)),
        "after_tax_p05": float(df["after_tax"].quantile(0.05)),
        "after_tax_p95": float(df["after_tax"].quantile(0.95)),
        "lifetime_tax_p50": float(df["total_lifetime_tax"].quantile(0.50)),
    })
comparison = pd.DataFrame(comparison_rows)
comparison["success_rate"] = comparison["success_rate"].map(lambda x: f"{x:.1%}")
comparison["after_tax_p05"] = comparison["after_tax_p05"].map(_fmt_money)
comparison["after_tax_p50"] = comparison["after_tax_p50"].map(_fmt_money)
comparison["after_tax_p95"] = comparison["after_tax_p95"].map(_fmt_money)
comparison["lifetime_tax_p50"] = comparison["lifetime_tax_p50"].map(_fmt_money)
display(comparison)

for strat in MC_STRATEGIES:
    name = strat["name"]
    _print_strategy_summary(name, mc_all[mc_all["strategy"] == name])

# ----------------------------
# 5) Plots (2 overlays + 3rd plot: strategy deltas)
# ----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Overlay outlines for readability
after_tax_bins = np.histogram_bin_edges(mc_all["after_tax"].astype(float), bins=60)
end_bins = np.histogram_bin_edges(mc_all["end_total_balance"].astype(float), bins=60)
colors = plt.cm.tab10.colors

for i, strat in enumerate(MC_STRATEGIES):
    name = strat["name"]
    df = mc_all[mc_all["strategy"] == name]
    c = colors[i % len(colors)]
    axes[0].hist(df["after_tax"], bins=after_tax_bins, histtype="step", linewidth=2.0, color=c, label=name)
    axes[1].hist(df["end_total_balance"], bins=end_bins, histtype="step", linewidth=2.0, color=c, label=name)
    axes[0].axvline(df["after_tax"].quantile(0.50), color=c, linestyle="--", linewidth=1.5)
    axes[1].axvline(df["end_total_balance"].quantile(0.50), color=c, linestyle="--", linewidth=1.5)

axes[0].set_title("After-tax wealth (MC) — dashed = median")
axes[0].set_xlabel("$")
axes[0].set_ylabel("count")
axes[0].legend(fontsize=8)

axes[1].set_title("Ending total balance (MC) — dashed = median")
axes[1].set_xlabel("$")
axes[1].set_ylabel("count")
axes[1].legend(fontsize=8)

# Third plot: distribution of after-tax differences vs Conservative
base_name = MC_STRATEGIES[0]["name"]
pivot = mc_all.pivot(index="sim", columns="strategy", values="after_tax")
deltas = pd.DataFrame({
    f"Aggressive - {base_name}": pivot[MC_STRATEGIES[1]["name"]] - pivot[base_name],
    f"Very Aggressive - {base_name}": pivot[MC_STRATEGIES[2]["name"]] - pivot[base_name],
})
delta_bins = np.histogram_bin_edges(deltas.stack().astype(float), bins=60)
axes[2].hist(deltas.iloc[:, 0], bins=delta_bins, histtype="step", linewidth=2.0, label=deltas.columns[0])
axes[2].hist(deltas.iloc[:, 1], bins=delta_bins, histtype="step", linewidth=2.0, label=deltas.columns[1])
axes[2].axvline(0.0, color="black", linestyle=":", linewidth=1.5)
axes[2].set_title("After-tax advantage vs Conservative")
axes[2].set_xlabel("$ (positive = better than Conservative)")
axes[2].set_ylabel("count")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()